<a href="https://colab.research.google.com/github/solven139/Quant-Analysis-Project/blob/main/Qaunt_Analysis_Trend_Filtered_Mean_Reversionipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install wrds

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 42.4 MB/s eta 0:00:00


In [ ]:
import wrds
import pandas as pd

UNIVERSE_TICKERS = ['DBC', 'EEM', 'EFA', 'GLD', 'HYG', 'IEF', 'IWM',
                    'LQD', 'QQQ', 'SHY', 'SLV', 'SPY', 'TLT', 'VNQ']

TARGET_DAYS = 30      # tenor point on the standardized surface
TARGET_DELTA_PUT = -25  # 25-delta put, stored as an integer percent in
                        # most WRDS OptionMetrics schemas (VERIFY: some
                        # installs store delta as a decimal, -0.25 --
                        # check a few sample rows before trusting this
                        # filter silently returns zero rows for the
                        # wrong reason).

db = wrds.Connection()

# ---------------------------------------------------------------------
# Step 1: resolve each of the 14 CRSP permnos (already known from this
# project's Phase 0b pull -- paste them in below once confirmed from
# `tri_full_panel_14assets_inception_summary.csv` or the original pull
# script's own permno resolution) to an OptionMetrics secid.
# ---------------------------------------------------------------------
# NOTE: fill in the 14 permnos from this project's existing Phase 0b
# pull before running -- left as a placeholder dict here since this
# script cannot query the sandbox's own already-pulled data.
PERMNO_MAP = {
    # 'SPY': 84398, 'QQQ': ..., ...  <- fill in from the existing pull
}

if not PERMNO_MAP:
    print("[STOP] Fill in PERMNO_MAP with this project's existing 14 permnos "
          "(from the Phase 0b pull) before running Step 1.")
else:
    permnos = tuple(PERMNO_MAP.values())
    link_query = f"""
        SELECT *
        FROM wrdsapps.opcrsphist
        WHERE permno IN {permnos}
    """
    # VERIFY the linking table name/columns above against your own WRDS
    # OptionMetrics schema browser before trusting this query -- if
    # `wrdsapps.opcrsphist` doesn't exist in your subscription, search
    # WRDS's own documentation for "CRSP OptionMetrics linking table."
    link_table = db.raw_sql(link_query)
    link_table.to_csv('optionshedge_dow_permno_secid_link.csv', index=False)
    print(f"Step 1 complete: {len(link_table)} permno-secid link rows saved.")

    secids = tuple(link_table['secid'].dropna().unique().tolist())

    # -------------------------------------------------------------
    # Step 2: pull the standardized Volatility Surface at the fixed
    # 30-day / 25-delta put point for every date and every resolved
    # secid. VERIFY the schema/table name (`optionm.vsurfd` is the most
    # common name at the time of writing, but WRDS has used
    # year-partitioned or `optionm_all`-prefixed variants historically)
    # and the delta sign/scale convention (integer percent vs decimal)
    # against a small sample query before pulling the full history.
    # -------------------------------------------------------------
    surface_query = f"""
        SELECT secid, date, days, delta, impl_volatility, impl_strike, impl_premium
        FROM optionm.vsurfd
        WHERE secid IN {secids}
          AND days = {TARGET_DAYS}
          AND delta = {TARGET_DELTA_PUT}
          AND cp_flag = 'P'
    """
    vol_surface = db.raw_sql(surface_query)
    vol_surface.to_csv('optionshedge_vol_surface.csv', index=False)
    print(f"Step 2 complete: {len(vol_surface)} volatility-surface rows saved.")

    print("\nStep 3 (plausibility check, run after both files are saved): "
          "confirm impl_volatility is in a sane annualized-decimal range "
          "(roughly 0.10-0.80 for these ETFs historically, with spikes to "
          "1.0+ during 2008/2020), confirm date coverage roughly matches "
          "this project's existing raw-price history per asset, and "
          "confirm no secid maps to more than one of the 14 tickers.")

db.close()

Enter your WRDS username [root]:lamnafur
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
[STOP] Fill in PERMNO_MAP with this project's existing 14 permnos (from the Phase 0b pull) before running Step 1.


In [ ]:
import wrds
import pandas as pd

SAMPLE_START = '1993-01-01'  # matches this project's existing 14-asset sample start
PRICE_LOOKBACK_BUFFER_DAYS = 400  # calendar days (~275 trading days) of extra
                                  # history pulled before each name's own
                                  # membership start, so its 200-trading-day
                                  # MA is already defined by the time it's
                                  # actually evaluated as a member

db = wrds.Connection()

# ---------------------------------------------------------------------
# Step 1: point-in-time S&P 500 membership, every spell overlapping
# [SAMPLE_START, today]. VERIFY the table/column names below against your
# own WRDS schema browser (see caveat 1 above) before trusting this query.
# ---------------------------------------------------------------------
membership_query = f"""
    SELECT permno, start, ending
    FROM crsp.dsp500list
    WHERE ending >= '{SAMPLE_START}'
"""
# If `crsp.dsp500list` doesn't exist in your subscription, search WRDS's
# documentation for the CIZ-format equivalent and adjust the table name
# and the `start`/`ending` column names (some variants use `mbrstartdt`/
# `mbrenddt` instead) before rerunning.
membership = db.raw_sql(membership_query, date_cols=['start', 'ending'])
membership.to_csv('sp500_membership_1993_present.csv', index=False)

n_permnos = membership['permno'].nunique()
print(f"Step 1 complete: {len(membership)} membership spells, "
      f"{n_permnos} distinct permnos, saved to sp500_membership_1993_present.csv")
print(f"  [plausibility] {n_permnos} distinct constituents over the sample window -- "
      f"expect roughly 1,500-2,500 given typical S&P 500 turnover; a number far "
      f"outside that range (e.g. under 600 or over 5,000) likely means the wrong "
      f"table/date-column convention was used above.")

# ---------------------------------------------------------------------
# Step 2: daily prices (raw price + split-adjustment factor, matching
# this project's already-established Phase C split-adjustment fix -- see
# breadth_data_prep.py, which will need the SAME dlyprc/dlycumfacpr split
# treatment as the options overlay's raw_price_panel did) for every
# distinct permno found in Step 1, each with a lookback buffer before its
# own earliest membership start date.
# ---------------------------------------------------------------------
permnos = tuple(membership['permno'].unique().tolist())
earliest_start_per_permno = membership.groupby('permno')['start'].min()
global_earliest_needed = (earliest_start_per_permno.min() -
                           pd.Timedelta(days=PRICE_LOOKBACK_BUFFER_DAYS))

price_query = f"""
    SELECT permno, dlycaldt, dlyprc, dlycumfacpr
    FROM crsp.dsf_v2
    WHERE permno IN {permnos}
      AND dlycaldt >= '{global_earliest_needed.date()}'
"""
# NOTE: this pulls each permno's FULL price history from the buffered
# start date through the present, not narrowly clipped to its own
# membership window -- simpler to query, and the "extra" pre/post-
# membership days are simply excluded downstream in breadth_data_prep.py
# by checking point-in-time membership before counting a name in that
# day's breadth denominator. If this single query is too large for your
# WRDS session, split `permnos` into batches of ~200 and concatenate the
# results before saving.
constituent_prices = db.raw_sql(price_query, date_cols=['dlycaldt'])
constituent_prices.to_csv('sp500_constituent_daily_prices_1993_present.csv', index=False)

print(f"\nStep 2 complete: {len(constituent_prices)} daily price rows across "
      f"{constituent_prices['permno'].nunique()} permnos, saved to "
      f"sp500_constituent_daily_prices_1993_present.csv")

print("\nStep 3 (plausibility check, run after both files are saved -- same "
      "discipline as every prior pull in this project, e.g. the seasonality-"
      "phase data integrity incident this project's own log records):")
print("  - On any given date, the number of DISTINCT permnos with active "
      "membership (start <= date <= ending) should be close to 500 (a "
      "small deviation, e.g. 498-505, is normal and expected; a value far "
      "from 500 across many dates suggests a membership-table problem).")
print("  - Spot-check a handful of long-tenured, well-known names (their "
      "permnos) appear as members continuously across most/all of the "
      "sample window.")
print("  - Confirm dlyprc is never NaN/non-positive and dlycumfacpr is "
      "never NaN/non-positive, matching the exact checks already applied "
      "to this project's original 14-asset raw price pull (optionshedge_"
      "dataprep.py's build_raw_price_panel).")

db.close()

Enter your WRDS username [root]:lamnafur
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Step 1 complete: 1308 membership spells, 1275 distinct permnos, saved to sp500_membership_1993_present.csv
  [plausibility] 1275 distinct constituents over the sample window -- expect roughly 1,500-2,500 given typical S&P 500 turnover; a number far outside that range (e.g. under 600 or over 5,000) likely means the wrong table/date-column convention was used above.

Step 2 complete: 12132066 daily price rows across 1275 permnos, saved to sp500_constituent_daily_prices_1993_present.csv

Step 3 (plausibility check, run after both files are saved -- same discipline as every prior pull in this project, e.g. the seasonality-phase data integrity incident this project's own log records):
  - On any g

In [ ]:
import os
import numpy as np
import pandas as pd
import wrds

SECTOR_TICKERS = ['XLE', 'XLF', 'XLK', 'XLV', 'XLI', 'XLP', 'XLY', 'XLU', 'XLB', 'XLRE']

# Known real-world inception dates, for the plausibility check at the end
# ONLY -- never used to filter or truncate anything pulled from WRDS.
KNOWN_INCEPTION = {
    'XLE': '1998-12-16', 'XLF': '1998-12-16', 'XLK': '1998-12-16',
    'XLV': '1998-12-16', 'XLI': '1998-12-16', 'XLP': '1998-12-16',
    'XLY': '1998-12-16', 'XLU': '1998-12-16', 'XLB': '1998-12-16',
    'XLRE': '2015-10-07',  # ~10 fewer years than the other nine -- kept anyway, flagged
}

PULL_START = '1998-01-01'  # a little before the earliest known inception, no later truncation

db = wrds.Connection()

# ---------------------------------------------------------------------
# Step 1: resolve each ticker to a CRSP permno. NO securitytype filter
# (see caveat 1 above) -- prints every candidate row so you can confirm
# the right permno per ticker before Step 2 runs.
# ---------------------------------------------------------------------
lookup_query = f"""
    SELECT permno, ticker, tradingsymbol, securitynm, issuernm,
           securitytype, securitysubtype, primaryexch,
           secinfostartdt, secinfoenddt, securityactiveflg
    FROM crsp.stksecurityinfohist
    WHERE ticker IN ({','.join(f"'{t}'" for t in SECTOR_TICKERS)})
    ORDER BY ticker, secinfostartdt
"""
lookup = db.raw_sql(lookup_query)
print(f"{len(lookup)} candidate rows found across {len(SECTOR_TICKERS)} tickers:\n")
print(lookup.to_string(index=False))

permno_map = {}
for t in SECTOR_TICKERS:
    sub = lookup[lookup['ticker'] == t]
    distinct = sub['permno'].unique()
    if len(distinct) == 1:
        permno_map[int(distinct[0])] = t
        print(f"\n{t}: exactly 1 distinct permno ({distinct[0]}) -- auto-resolved.")
    else:
        print(f"\n!! {t}: {len(distinct)} distinct permno(s) found ({list(distinct)}) -- "
              f"NOT auto-resolved. Inspect the printed rows above (securitytype/"
              f"securitysubtype/date ranges) and add the correct one to permno_map "
              f"by hand below before continuing.")

# --- If any ticker was flagged above, resolve it by hand here before Step 2, e.g.: ---
# permno_map[XXXXX] = 'XLRE'

print(f"\nResolved {len(permno_map)} / {len(SECTOR_TICKERS)} tickers.")
missing = set(SECTOR_TICKERS) - set(permno_map.values())
if missing:
    print(f"[STOP] Still unresolved: {missing} -- fix permno_map by hand before running Step 2.")

# ---------------------------------------------------------------------
# Step 2: daily prices -- SAME columns as this project's existing
# `crsp_daily_raw_combined_14assets.csv`, from the same table
# (crsp.dsf_v2), so downstream code needs no changes.
# ---------------------------------------------------------------------
if permno_map and not missing:
    permnos = list(permno_map.keys())
    price_query = f"""
        SELECT permno, dlycaldt, dlyprc, dlyclose, dlyopen, dlyhigh, dlylow,
               dlyret, dlyretx, dlyvol, dlycumfacpr, dlycumfacshr,
               dlydelflg, tradingstatusflg
        FROM crsp.dsf_v2
        WHERE permno IN ({','.join(str(p) for p in permnos)})
          AND dlycaldt >= '{PULL_START}'
        ORDER BY permno, dlycaldt
    """
    raw = db.raw_sql(price_query, date_cols=['dlycaldt'])
    raw['ticker'] = raw['permno'].map(permno_map)
    raw['asset'] = raw['ticker']

    os.makedirs('data/raw', exist_ok=True)
    raw.to_csv('data/raw/sector_etf_crsp_daily_raw_combined.csv', index=False)
    print(f"\nStep 2 complete: {len(raw)} daily rows across {raw['ticker'].nunique()} "
          f"tickers, saved to data/raw/sector_etf_crsp_daily_raw_combined.csv")

    # -------------------------------------------------------------
    # Step 3: build the two final panels, using this project's own
    # already-validated conventions exactly (see optionshedge_dataprep.py
    # for the identical logic applied to the 14-asset universe):
    #   - raw (split-adjusted) price panel: dlyprc / dlycumfacpr
    #   - TRI panel: per-asset cumulative product of (1 + dlyret),
    #     rebased to 1.0 on EACH asset's own first available date
    #     (per-asset-independent staggered inception, never truncated to
    #     the latest-starting asset -- same standing convention as the
    #     main 14-asset universe).
    # -------------------------------------------------------------
    assert raw['dlyprc'].isna().sum() == 0, "Unexpected NaN dlyprc"
    assert (raw['dlyprc'] <= 0).sum() == 0, "Unexpected non-positive dlyprc"
    assert raw['dlycumfacpr'].isna().sum() == 0, "Unexpected NaN dlycumfacpr"
    assert (raw['dlycumfacpr'] <= 0).sum() == 0, "Unexpected non-positive dlycumfacpr"

    raw['split_adj_price'] = raw['dlyprc'] / raw['dlycumfacpr']
    raw_price_panel = raw.pivot(index='dlycaldt', columns='ticker', values='split_adj_price')
    raw_price_panel = raw_price_panel[SECTOR_TICKERS]
    raw_price_panel.index.name = 'date'
    raw_price_panel.to_csv('data/raw/raw_price_panel_10sectors.csv')
    print(f"Saved: data/raw/raw_price_panel_10sectors.csv, shape {raw_price_panel.shape}")

    ret_panel = raw.pivot(index='dlycaldt', columns='ticker', values='dlyret')
    ret_panel = ret_panel[SECTOR_TICKERS]
    # ANCHOR DATE, verified directly against the real 14-asset panel before
    # trusting this: it must be each ticker's own FIRST ROW IN THE RAW PULL
    # (its actual inception date), NOT `ret_panel[t].dropna().index[0]`.
    # dlyret is NaN on the inception date itself (no prior day to compare
    # against), so dropna()'s first index is one trading day LATE -- an
    # easy-to-miss off-by-one that silently drops the first day's implicit
    # "rebase to 1.0" step and shifts every later value by that one day's
    # compounding factor, forever. Caught here by reconstructing this
    # project's OWN existing 14-asset TRI panel with this exact code and
    # diffing against the real file: the dropna()-based anchor was off by
    # a full trading day for all 14 tickers (max abs diff up to ~4.7 TRI
    # units by the end of a 30-year series); anchoring on the raw pull's
    # own first row instead reproduces the real file to float precision
    # (~1e-15) for 13 of 14. The single remaining mismatch (QQQ) traced to
    # a genuine multi-day GAP in this project's existing raw pull around
    # 2004-12-01 -- the week Nasdaq-100 Trust Series 1 moved its listing
    # from AMEX to Nasdaq (ticker QQQ -> QQQQ) -- a real identifier-
    # continuity event, the same class of issue as this project's own
    # earlier "secid reuse"/ticker-reuse bugs, NOT a flaw in this
    # transform. See the trading-day-gap check below, added specifically
    # because of what this diagnosis surfaced.
    first_date_per_ticker = raw.groupby('ticker')['dlycaldt'].min()
    tri_panel = pd.DataFrame(index=ret_panel.index, columns=SECTOR_TICKERS, dtype=float)
    for t in SECTOR_TICKERS:
        if t not in first_date_per_ticker.index:
            print(f"!! {t}: no rows found in the raw pull at all -- check the pull.")
            continue
        first_date = first_date_per_ticker[t]
        sub_idx = ret_panel.index[ret_panel.index >= first_date]
        rets_from_first = ret_panel.loc[sub_idx, t].fillna(0.0)
        tri_panel.loc[sub_idx, t] = (1.0 + rets_from_first).cumprod()
    tri_panel.index.name = 'dlycaldt'
    tri_panel.to_csv('data/raw/tri_full_panel_10sectors.csv')
    print(f"Saved: data/raw/tri_full_panel_10sectors.csv, shape {tri_panel.shape}")

    # -------------------------------------------------------------
    # Trading-day-GAP check, added specifically because diagnosing the
    # QQQ mismatch above surfaced a real, silent failure mode: a ticker
    # can have a multi-day hole INSIDE its own active window (e.g. an
    # exchange-listing change) that no NaN-count check catches, because
    # the row for that day is simply ABSENT, not present-with-NaN. Using
    # SPY's own date index as the reference trading calendar (SPY has the
    # longest, most complete history of any asset in this project and has
    # never had this kind of gap, confirmed above) flags any sector ETF
    # missing a trading day inside its own [first, last] window.
    # -------------------------------------------------------------
    print("\n--- Trading-day-gap check (missing rows inside a ticker's own active window) ---")
    # SPY is not part of this pull, so the reference calendar here is the
    # UNION of all 10 sector tickers' own trading dates -- any date traded
    # by at least one other sector ETF but missing from a given ticker's
    # own [first, last] window is a candidate gap worth a manual look.
    all_dates = raw['dlycaldt'].sort_values().unique()
    for t in SECTOR_TICKERS:
        if t not in first_date_per_ticker.index:
            continue
        own_first, own_last = first_date_per_ticker[t], raw[raw['ticker'] == t]['dlycaldt'].max()
        expected_dates = all_dates[(all_dates >= np.datetime64(own_first)) & (all_dates <= np.datetime64(own_last))]
        own_dates = set(raw[raw['ticker'] == t]['dlycaldt'])
        missing = sorted(set(expected_dates) - own_dates)
        if missing:
            preview = [pd.Timestamp(d).date().isoformat() for d in missing[:10]]
            print(f"  !! {t}: {len(missing)} date(s) traded by other sector ETFs but MISSING for "
                  f"{t} inside its own [{own_first.date()}, {own_last.date()}] window -- "
                  f"inspect by hand (first few: {preview}). This is exactly the QQQ/2004 "
                  f"failure mode diagnosed above; do not assume it's benign.")
        else:
            print(f"  {t}: no gaps found.")

    # -------------------------------------------------------------
    # Step 4: plausibility checks -- same discipline as every prior pull.
    # -------------------------------------------------------------
    print("\n--- Plausibility checks ---")
    for t in SECTOR_TICKERS:
        first_real = tri_panel[t].dropna().index.min()
        expected = pd.Timestamp(KNOWN_INCEPTION[t])
        gap_days = abs((first_real - expected).days) if pd.notna(first_real) else None
        flag = "" if (gap_days is not None and gap_days <= 10) else "  !! CHECK THIS"
        print(f"  {t}: first real TRI date = {first_real.date() if pd.notna(first_real) else 'MISSING'}, "
              f"expected ~{expected.date()}{flag}")
    print(f"\n  TRI value on each ticker's own first date should be exactly 1.000000 "
          f"(rebased there, per-asset independent):")
    print(f"  {tri_panel.apply(lambda c: c.dropna().iloc[0] if c.notna().any() else np.nan).round(6).to_dict()}")

    db.close()
    print("\nOnce this runs clean, send back the two panel files: "
          "raw_price_panel_10sectors.csv and tri_full_panel_10sectors.csv "
          "(the raw combined file is optional -- only needed if something "
          "looks wrong and needs debugging from the source rows).")
else:
    db.close()

Enter your WRDS username [root]:lamnafur
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
66 candidate rows found across 10 tickers:

 permno ticker tradingsymbol                              securitynm                 issuernm securitytype securitysubtype primaryexch secinfostartdt secinfoenddt securityactiveflg
  86449    XLB          <NA> SELECT SECTOR SPDR TRUST; ETFNONE; CONS SELECT SECTOR SPDR TRUST         FUND             ETF           A     1998-12-22   2002-01-01                 Y
  86449    XLB           XLB SELECT SECTOR SPDR TRUST; ETFNONE; CONS SELECT SECTOR SPDR TRUST         FUND             ETF           A     2002-01-02   2004-06-09                 Y
  86449    XLB           XLB SELECT SECTOR SPDR TRUST; ETFNONE; CONS SELECT SECTOR SPDR TRUST         FUND        

In [ ]:
import numpy as np
import pandas as pd

TRI_PATH = 'data/raw/tri_full_panel_14assets.csv'
YIELD_PATH = 'data/raw/treasury_yields_fred.csv'

VOL_SHORT_WINDOW = 21
VOL_LONG_WINDOW = 252
TREND_LOOKBACK = 252
CREDIT_WINDOW = 60           # user's pre-registered spec (Phase F used 63)
SADF_LOOKBACK = 63           # unchanged from step3b_metalabel_final_feature_round.py
SADF_MIN_SAMPLE = 20
SADF_LAGS = 1
MIN_WARMUP_DAYS = 252        # calendar-day-since-inception gate, all 5 conditions

DAILY_CONDITIONS = ['vol_regime', 'trend_strength', 'credit_spread', 'rate_environment']
ALL_CONDITIONS = DAILY_CONDITIONS + ['structural_break']  # + market_breadth once available


# ==========================================================================
# The four DAILY conditions -- one per-asset DataFrame each, same shape as
# price_panel, NaN until each condition's own warmup window is satisfied.
# ==========================================================================

def compute_vol_regime_panel(price_panel):
    """Per-asset trailing 21d/252d realized-vol ratio (own returns)."""
    ret = price_panel.pct_change(fill_method=None)
    vol_short = ret.rolling(VOL_SHORT_WINDOW, min_periods=VOL_SHORT_WINDOW).std()
    vol_long = ret.rolling(VOL_LONG_WINDOW, min_periods=VOL_LONG_WINDOW).std()
    return vol_short / vol_long


def compute_trend_strength_panel(price_panel):
    """Per-asset |trailing 252-day return| (own price)."""
    return price_panel.pct_change(TREND_LOOKBACK, fill_method=None).abs()


def compute_credit_spread_series(price_panel):
    """Market-wide HYG-minus-LQD trailing 60-day return differential."""
    for col in ['HYG', 'LQD']:
        assert col in price_panel.columns, f"compute_credit_spread_series requires column {col}"
    hyg_ret = price_panel['HYG'].pct_change(CREDIT_WINDOW, fill_method=None)
    lqd_ret = price_panel['LQD'].pct_change(CREDIT_WINDOW, fill_method=None)
    return hyg_ret - lqd_ret


def compute_rate_environment_series(yields_df):
    """
    Market-wide DGS10-minus-DGS3MO LEVEL, on FRED's own calendar (no
    forward-fill -- exact-date lookup only, same no-ffill discipline as
    carry_signal.py's own DGS20/DGS3MO handling).
    """
    dgs10 = pd.to_numeric(yields_df['DGS10'], errors='coerce')
    dgs3mo = pd.to_numeric(yields_df['DGS3MO'], errors='coerce')
    return (dgs10 - dgs3mo).dropna()


def build_daily_condition_panels(price_panel, yields_df):
    """
    Returns dict: condition name -> DataFrame (same shape/columns as
    price_panel). The two market-wide conditions (credit_spread,
    rate_environment) are broadcast to every column, then masked to NaN
    outside each asset's own active window (its own first/last valid
    price date) so the per-asset warmup/median logic below "sees" each
    asset's own history length, not the panel's.
    """
    active_mask = price_panel.notna()
    # forward/backward-fill-free active window per asset: True from that
    # asset's own first valid date through its own last valid date.
    first_valid = price_panel.apply(lambda s: s.first_valid_index())
    last_valid = price_panel.apply(lambda s: s.last_valid_index())
    window_mask = pd.DataFrame(False, index=price_panel.index, columns=price_panel.columns)
    for col in price_panel.columns:
        if first_valid[col] is None:
            continue
        window_mask.loc[first_valid[col]:last_valid[col], col] = True

    vol_regime = compute_vol_regime_panel(price_panel).where(window_mask)
    trend_strength = compute_trend_strength_panel(price_panel).where(window_mask)

    credit_spread_1d = compute_credit_spread_series(price_panel)
    credit_spread = pd.DataFrame(
        {col: credit_spread_1d for col in price_panel.columns}, index=price_panel.index
    ).where(window_mask)

    rate_env_1d = compute_rate_environment_series(yields_df).reindex(price_panel.index)
    rate_environment = pd.DataFrame(
        {col: rate_env_1d for col in price_panel.columns}, index=price_panel.index
    ).where(window_mask)

    return {
        'vol_regime': vol_regime,
        'trend_strength': trend_strength,
        'credit_spread': credit_spread,
        'rate_environment': rate_environment,
    }


# ==========================================================================
# Point-in-time expanding-window per-asset median split (the core
# machinery, shared by all conditions -- daily or sparse).
# ==========================================================================

def expanding_median_label(daily_panel, min_warmup_days=MIN_WARMUP_DAYS):
    """
    daily_panel : DataFrame, DatetimeIndex, one column per asset -- a
        DAILY condition series already masked to each asset's own active
        window (NaN outside it, e.g. build_daily_condition_panels' output).

    Returns a same-shaped DataFrame of labels in {+1.0 (HIGH), -1.0 (LOW),
    NaN}. For asset A at row position i within A's own active window
    (i=0 at A's own first valid date), the label uses the expanding
    median of A's own valid values at positions [0, i-1] (STRICTLY prior
    -- today's own value never contributes to today's own threshold), and
    is NaN until i >= min_warmup_days, i.e. at least one full year of A's
    own calendar history has elapsed since ITS OWN inception.
    """
    out = pd.DataFrame(np.nan, index=daily_panel.index, columns=daily_panel.columns)
    for col in daily_panel.columns:
        s = daily_panel[col]
        first = s.first_valid_index()
        if first is None:
            continue
        window = s.loc[first:]
        vals = window.to_numpy()
        n = len(vals)
        # expanding median of strictly-prior valid values, via shift(1)
        prior_median = pd.Series(vals).shift(1).expanding(min_periods=1).median().to_numpy()
        labels = np.full(n, np.nan)
        for i in range(n):
            if i < min_warmup_days:
                continue
            if np.isnan(vals[i]) or np.isnan(prior_median[i]):
                continue
            labels[i] = 1.0 if vals[i] >= prior_median[i] else -1.0
        out.loc[window.index, col] = labels
    return out


def expanding_median_label_sparse(sparse_values_by_asset, inception_by_asset,
                                   min_warmup_days=MIN_WARMUP_DAYS):
    """
    Sparse-history variant, for structural_break (see module docstring's
    flagged compute-cost simplification). sparse_values_by_asset: dict of
    asset -> Series (DatetimeIndex = the sparse set of dates the condition
    was actually evaluated at for that asset, e.g. trade decision dates,
    sorted ascending). inception_by_asset: dict of asset -> that asset's
    own TRUE first valid price date (used for the calendar-day warmup gate
    -- NOT the first sparse-evaluation date, which is typically much later
    than inception).

    Returns dict: asset -> Series of labels in {+1.0, -1.0, NaN}, same
    index as that asset's own sparse_values series.
    """
    out = {}
    for asset, s in sparse_values_by_asset.items():
        s = s.sort_index()
        inception = inception_by_asset[asset]
        vals = s.to_numpy()
        n = len(vals)
        prior_median = pd.Series(vals).shift(1).expanding(min_periods=1).median().to_numpy()
        labels = np.full(n, np.nan)
        for i in range(n):
            days_since_inception = (s.index[i] - inception).days
            if days_since_inception < min_warmup_days:
                continue
            if np.isnan(vals[i]) or np.isnan(prior_median[i]):
                continue
            labels[i] = 1.0 if vals[i] >= prior_median[i] else -1.0
        out[asset] = pd.Series(labels, index=s.index)
    return out


# ==========================================================================
# Structural break (SADF) -- sparse, reused verbatim from
# step3b_metalabel_final_feature_round.py (unchanged math).
# ==========================================================================

def get_beta_parts(X, y):
    xy = X.T @ y
    xx = X.T @ X
    xx_inv = np.linalg.pinv(xx)
    return xx, xx_inv, xy


def get_betas(X, y):
    xx, xx_inv, xy = get_beta_parts(X, y)
    beta = xx_inv @ xy
    err = y - X @ beta
    beta_var = err.T @ err / (X.shape[0] - X.shape[1]) * xx_inv
    beta_std = np.sqrt(np.diag(beta_var))
    return beta, beta_std


def lag_x(x, lags):
    if isinstance(lags, int):
        lags = range(lags + 1)
    return np.concatenate([x[(np.max(lags) - lag):(len(x) - lag)].reshape(-1, 1) for lag in lags], axis=1)


def np_get_xy(log_price, regression, lags):
    log_rtn = log_price[1:] - log_price[:-1]
    X = lag_x(log_rtn, lags)
    n_skip = log_price.shape[0] - X.shape[0]
    y = log_rtn[n_skip - 1:]
    X[:, 0] = log_price[n_skip - 1:-1]
    if regression != 'n':
        X = np.append(X, np.ones((X.shape[0], 1)), axis=1)
    if regression[:2] == 'ct':
        trend = np.arange(X.shape[0]).reshape(-1, 1)
        X = np.append(X, trend, axis=1)
    if regression == 'ctt':
        X = np.append(X, trend ** 2, axis=1)
    return X, y


def get_bsadf(log_price, min_sample, regression='c', lags=3):
    log_price = np.asarray(log_price)
    X, y = np_get_xy(log_price, regression=regression, lags=lags)
    min_sample = max([int(2 * np.max(lags) + 9), min_sample])
    start_points = range(0, y.shape[0] - min_sample + 1)
    adf_vals = np.zeros(len(start_points))
    for start in start_points:
        X_sub, y_sub = X[start:], y[start:]
        beta, beta_error = get_betas(X_sub, y_sub)
        adf_vals[start] = beta[0] / beta_error[0]
    return np.nanquantile(adf_vals, 0.99)


def compute_structural_break_at_dates(price_series, dates, lookback=SADF_LOOKBACK,
                                       min_sample=SADF_MIN_SAMPLE, lags=SADF_LAGS):
    """
    price_series : Series, DatetimeIndex, one asset's own TRI level.
    dates : iterable of Timestamps to evaluate at (must be in price_series's
        own index; each needs `lookback` trailing valid observations).

    Returns a Series indexed by `dates` (dates without enough trailing
    history, or not present in price_series, are NaN).
    """
    log_price = np.log(price_series)
    out = {}
    for d in dates:
        if d not in log_price.index:
            out[d] = np.nan
            continue
        pos = log_price.index.get_loc(d)
        if pos + 1 < lookback:
            out[d] = np.nan
            continue
        window = log_price.iloc[pos - lookback + 1: pos + 1].to_numpy()
        if np.isnan(window).any():
            out[d] = np.nan
            continue
        out[d] = get_bsadf(window, min_sample=min_sample, regression='c', lags=lags)
    return pd.Series(out)


# ==========================================================================
# Market breadth -- DEFERRED placeholder (Step 3, blocked on the real
# wrds_breadth_pull.py run). Wired in now so it plugs into the exact same
# expanding_median_label() machinery once real data exists.
# ==========================================================================

def market_breadth_placeholder(price_panel):
    """
    Returns an all-NaN DataFrame, same shape as price_panel. Once the real
    pct_above_200dma panel (from breadth_data_prep.py, run on
    wrds_breadth_pull.py's real output) is available, replace the body of
    this function with that panel (broadcast + active-window-masked, same
    pattern as compute_credit_spread_series/compute_rate_environment_series
    above) -- expanding_median_label() itself needs no change.
    """
    return pd.DataFrame(np.nan, index=price_panel.index, columns=price_panel.columns)


# ==========================================================================
# Orchestration: attach all 5 available condition HIGH/LOW labels to a
# bets/trades DataFrame (condedge_bet_dataset.extract_trades_for_asset's
# own output schema: one row per completed round-trip trade, with `asset`
# and `decision_date` columns) at each trade's own decision_date.
# ==========================================================================

def label_trades_by_condition(bets, price_panel, yields_df):
    """
    bets : DataFrame with columns 'asset', 'decision_date' (and whatever
        else condedge_bet_dataset.build_bet_dataset-style trade extraction
        produced -- this function only reads those two columns).

    Returns a copy of `bets` with 5 new columns, one per available
    condition (vol_regime, trend_strength, credit_spread, rate_environment,
    structural_break), each in {+1.0 (HIGH), -1.0 (LOW), NaN}.
    """
    bets = bets.copy()
    daily_panels = build_daily_condition_panels(price_panel, yields_df)
    daily_labels = {name: expanding_median_label(panel) for name, panel in daily_panels.items()}

    for name in DAILY_CONDITIONS:
        lbl_panel = daily_labels[name]
        vals = np.full(len(bets), np.nan)
        for i, (asset, d) in enumerate(zip(bets['asset'], bets['decision_date'])):
            if asset in lbl_panel.columns and d in lbl_panel.index:
                vals[i] = lbl_panel.at[d, asset]
        bets[name] = vals

    # structural_break: sparse, one asset at a time, only at that asset's
    # own set of decision dates actually present in bets.
    first_valid = price_panel.apply(lambda s: s.first_valid_index())
    sb_labels_all = pd.Series(np.nan, index=bets.index)
    for asset, rows in bets.groupby('asset').groups.items():
        if asset not in price_panel.columns or first_valid[asset] is None:
            continue
        dates_needed = sorted(set(bets.loc[rows, 'decision_date']))
        sadf_vals = compute_structural_break_at_dates(price_panel[asset], dates_needed)
        sadf_labels = expanding_median_label_sparse(
            {asset: sadf_vals}, {asset: first_valid[asset]})[asset]
        for i in rows:
            d = bets.at[i, 'decision_date']
            if d in sadf_labels.index:
                sb_labels_all.at[i] = sadf_labels.at[d]
    bets['structural_break'] = sb_labels_all.values

    return bets


# === RUN_FROM_HERE ===

if __name__ == '__main__':
    import os
    import sys
    # Robust to BOTH execution styles: `!python condition_features.py` (where
    # __file__ is this script's own real path) AND pasting/running this file
    # as a Colab/Jupyter cell (where some kernels set __file__ to a USELESS
    # transient path like /tmp/ipykernel_.../<cell>.py, which silently sends
    # the old single-branch version of this line looking in the wrong
    # directory for sibling modules -- a real failure mode hit in practice,
    # not a hypothetical one). Try cwd FIRST (correct for a Colab cell,
    # where uploaded files land in the working directory), then __file__'s
    # own directory as a fallback (correct for a real script invocation).
    for _d in dict.fromkeys(
        [os.getcwd()] + ([os.path.dirname(os.path.abspath(__file__))] if '__file__' in dir() else [])
    ):
        if _d not in sys.path:
            sys.path.insert(0, _d)
    from trend_meanrev_signal import build_trend_filtered_signal
    from condedge_bet_dataset import extract_trades_for_asset

    price_panel = pd.read_csv(TRI_PATH, index_col=0, parse_dates=True)
    yields_df = pd.read_csv(YIELD_PATH, index_col=0, parse_dates=True)

    executed_position_panel = build_trend_filtered_signal(price_panel)
    daily_index = price_panel.index
    simple_ret_panel = price_panel.pct_change(fill_method=None)

    all_rows = []
    for asset in price_panel.columns:
        all_rows.extend(extract_trades_for_asset(
            asset, executed_position_panel[asset], simple_ret_panel[asset], daily_index))
    bets = pd.DataFrame(all_rows)
    bets = bets[bets['decision_date'].notna()].sort_values('t0').reset_index(drop=True)
    print(f"Total completed round-trip trades (14-asset universe): {len(bets)}")

    import time
    t0 = time.time()
    labeled = label_trades_by_condition(bets, price_panel, yields_df)
    print(f"Condition labeling completed in {time.time() - t0:.1f}s")

    pd.set_option('display.width', 160)
    print("\n--- HIGH/LOW group sizes per condition (NaN = warmup not cleared / missing input) ---")
    for cond in ALL_CONDITIONS:
        counts = labeled[cond].value_counts(dropna=False)
        n_high = counts.get(1.0, 0)
        n_low = counts.get(-1.0, 0)
        n_nan = counts.get(np.nan, 0) if labeled[cond].isna().any() else 0
        print(f"  {cond:18s}  HIGH={n_high:5d}  LOW={n_low:5d}  NaN={n_nan:5d}")

    os.makedirs('data/raw', exist_ok=True)
    labeled.to_csv('data/raw/condedge_trades_labeled_6cond_14assets.csv', index=False)
    print("\nSaved: data/raw/condedge_trades_labeled_6cond_14assets.csv "
          "(market_breadth column not yet included -- Step 3, pending real breadth data)")

Total completed round-trip trades (14-asset universe): 1309
Condition labeling completed in 5.8s

--- HIGH/LOW group sizes per condition (NaN = warmup not cleared / missing input) ---
  vol_regime          HIGH=  629  LOW=  635  NaN=   45
  trend_strength      HIGH=  431  LOW=  833  NaN=   45
  credit_spread       HIGH=  528  LOW=  515  NaN=  266
  rate_environment    HIGH=  397  LOW=  899  NaN=   13
  structural_break    HIGH=  613  LOW=  682  NaN=   14

Saved: data/raw/condedge_trades_labeled_6cond_14assets.csv (market_breadth column not yet included -- Step 3, pending real breadth data)


In [ ]:
import os
print("cwd:", os.getcwd())
!ls -la /content/data/raw/

cwd: /content
total 10124
drwxr-xr-x 3 root root    4096 Sep 30 13:30 .
drwxr-xr-x 3 root root    4096 Sep 30 11:52 ..
-rw-r--r-- 1 root root   27916 Sep 30 13:31 condedge_dsr_combined_verdict_14assets.csv
-rw-r--r-- 1 root root   15111 Sep 30 13:31 condedge_dsr_trial_inputs.csv
-rw-r--r-- 1 root root   14210 Sep 30 13:12 condedge_pricepath_mcpt_results_14assets.csv
-rw-r--r-- 1 root root  116088 Sep 30 13:35 condedge_trades_labeled_6cond_14assets.csv
drwxr-xr-x 2 root root    4096 Sep 30 12:14 .ipynb_checkpoints
-rw-r--r-- 1 root root  543686 Sep 30 11:52 raw_price_panel_10sectors.csv
-rw-r--r-- 1 root root 6191902 Sep 30 11:52 sector_etf_crsp_daily_raw_combined.csv
-rw-r--r-- 1 root root  517830 Sep 30 12:30 treasury_yields_fred.csv
-rw-r--r-- 1 root root 1260370 Sep 30 11:52 tri_full_panel_10sectors.csv
-rw-r--r-- 1 root root 1652795 Sep 30 12:24 tri_full_panel_14assets.csv


In [ ]:
import wrds
import pandas as pd

SAMPLE_START = '1993-01-01'  # matches this project's existing 14-asset sample start
PRICE_LOOKBACK_BUFFER_DAYS = 400  # calendar days (~275 trading days) of extra
                                  # history pulled before each name's own
                                  # membership start, so its 200-trading-day
                                  # MA is already defined by the time it's
                                  # actually evaluated as a member

db = wrds.Connection()

# ---------------------------------------------------------------------
# Step 1: point-in-time S&P 500 membership, every spell overlapping
# [SAMPLE_START, today]. VERIFY the table/column names below against your
# own WRDS schema browser (see caveat 1 above) before trusting this query.
# ---------------------------------------------------------------------
membership_query = f"""
    SELECT permno, start, ending
    FROM crsp.dsp500list
    WHERE ending >= '{SAMPLE_START}'
"""
# If `crsp.dsp500list` doesn't exist in your subscription, search WRDS's
# documentation for the CIZ-format equivalent and adjust the table name
# and the `start`/`ending` column names (some variants use `mbrstartdt`/
# `mbrenddt` instead) before rerunning.
membership = db.raw_sql(membership_query, date_cols=['start', 'ending'])
membership.to_csv('sp500_membership_1993_present.csv', index=False)

n_permnos = membership['permno'].nunique()
print(f"Step 1 complete: {len(membership)} membership spells, "
      f"{n_permnos} distinct permnos, saved to sp500_membership_1993_present.csv")
print(f"  [plausibility] {n_permnos} distinct constituents over the sample window -- "
      f"expect roughly 1,500-2,500 given typical S&P 500 turnover; a number far "
      f"outside that range (e.g. under 600 or over 5,000) likely means the wrong "
      f"table/date-column convention was used above.")

# ---------------------------------------------------------------------
# Step 2: daily prices (raw price + split-adjustment factor, matching
# this project's already-established Phase C split-adjustment fix -- see
# breadth_data_prep.py, which will need the SAME dlyprc/dlycumfacpr split
# treatment as the options overlay's raw_price_panel did) for every
# distinct permno found in Step 1, each with a lookback buffer before its
# own earliest membership start date.
# ---------------------------------------------------------------------
permnos = tuple(membership['permno'].unique().tolist())
earliest_start_per_permno = membership.groupby('permno')['start'].min()
global_earliest_needed = (earliest_start_per_permno.min() -
                           pd.Timedelta(days=PRICE_LOOKBACK_BUFFER_DAYS))

price_query = f"""
    SELECT permno, dlycaldt, dlyprc, dlycumfacpr
    FROM crsp.dsf_v2
    WHERE permno IN {permnos}
      AND dlycaldt >= '{global_earliest_needed.date()}'
"""
# NOTE: this pulls each permno's FULL price history from the buffered
# start date through the present, not narrowly clipped to its own
# membership window -- simpler to query, and the "extra" pre/post-
# membership days are simply excluded downstream in breadth_data_prep.py
# by checking point-in-time membership before counting a name in that
# day's breadth denominator. If this single query is too large for your
# WRDS session, split `permnos` into batches of ~200 and concatenate the
# results before saving.
constituent_prices = db.raw_sql(price_query, date_cols=['dlycaldt'])
constituent_prices.to_csv('sp500_constituent_daily_prices_1993_present.csv', index=False)

print(f"\nStep 2 complete: {len(constituent_prices)} daily price rows across "
      f"{constituent_prices['permno'].nunique()} permnos, saved to "
      f"sp500_constituent_daily_prices_1993_present.csv")

print("\nStep 3 (plausibility check, run after both files are saved -- same "
      "discipline as every prior pull in this project, e.g. the seasonality-"
      "phase data integrity incident this project's own log records):")
print("  - On any given date, the number of DISTINCT permnos with active "
      "membership (start <= date <= ending) should be close to 500 (a "
      "small deviation, e.g. 498-505, is normal and expected; a value far "
      "from 500 across many dates suggests a membership-table problem).")
print("  - Spot-check a handful of long-tenured, well-known names (their "
      "permnos) appear as members continuously across most/all of the "
      "sample window.")
print("  - Confirm dlyprc is never NaN/non-positive and dlycumfacpr is "
      "never NaN/non-positive, matching the exact checks already applied "
      "to this project's original 14-asset raw price pull (optionshedge_"
      "dataprep.py's build_raw_price_panel).")

db.close()

Enter your WRDS username [root]:lamnafur
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Step 1 complete: 1308 membership spells, 1275 distinct permnos, saved to sp500_membership_1993_present.csv
  [plausibility] 1275 distinct constituents over the sample window -- expect roughly 1,500-2,500 given typical S&P 500 turnover; a number far outside that range (e.g. under 600 or over 5,000) likely means the wrong table/date-column convention was used above.

Step 2 complete: 12132066 daily price rows across 1275 permnos, saved to sp500_constituent_daily_prices_1993_present.csv

Step 3 (plausibility check, run after both files are saved -- same discipline as every prior pull in this project, e.g. the seasonality-phase data integrity incident this project's own log records):
  - On any g

In [ ]:
import numpy as np
import pandas as pd

MA_WINDOW = 200
MEMBERSHIP_PATH = 'sp500_membership_1993_present.csv'
PRICES_PATH = 'sp500_constituent_daily_prices_1993_present.csv'


def load_and_validate_prices(prices_path=PRICES_PATH):
    """
    *** REAL-DATA FIX (found once run against the user's actual WRDS pull,
    not assumed in advance -- flagged here exactly like optionshedge_
    dataprep.py's own split-adjustment fix was for the 14-asset universe;
    REVISED a second time after the user's own real pull showed
    `dlycumfacpr` is not the clean field this function first assumed
    either -- see point (c) below) ***

    The original version of this function asserted `dlyprc` AND
    `dlycumfacpr` were always present and positive -- true in practice for
    the 14-asset universe's own tiny, hand-picked, highly-liquid tickers
    (optionshedge_dataprep.py's identical assertions DID hold there), but
    NOT for a ~1,500-2,500-constituent S&P 500 history over 30+ years,
    which genuinely includes thinly-traded/halted/gap days for less-liquid
    names -- on those days CRSP can leave EITHER or BOTH fields null for a
    given (permno, date) row, not just `dlyprc` alone. Three real CRSP
    patterns, all handled the same way -- produce a NaN `split_adj_price`
    for that one observation, never a crash, never a fabricated value --
    rather than asserting the pull is clean in advance:

      (a) NaN dlyprc -- no price/quote at all that day for that permno.
      (b) Negative dlyprc -- CRSP's own long-standing convention meaning
          "no trade occurred that day; this is a bid-ask-average estimate,"
          NOT a data error. Recovered via abs() -- a real, if lower-
          quality, price estimate. Dropping every no-trade day instead
          would bias thinly-traded (typically smaller) names out of the
          breadth count more than liquid ones, exactly the kind of
          selection bias this project avoids elsewhere. Exact zero IS a
          genuine invalid value (a price cannot be zero) and is converted
          to NaN, not kept as a real observation -- same treatment as (a).
      (c) NaN or non-positive dlycumfacpr -- this function originally
          assumed this field would always be clean (the user's first real
          run proved that assumption wrong: it hit an "Unexpected NaN
          dlycumfacpr" crash on the very next line after (a)/(b) were
          fixed). Without a valid adjustment factor there is no principled
          way to split-adjust that day's price, so it is treated exactly
          like (a) -- set to NaN, never guessed/filled/interpolated (this
          project's own standing no-silent-patching discipline) -- rather
          than dividing by a missing or non-positive number, which would
          produce inf or a wrongly-signed nonsense value.

    In every case, the missing observation flows through as NaN in
    `split_adj_price` -- downstream compute_above_200dma_panel/
    compute_breadth_series already treat a missing observation as "not
    eligible that day" via their own existing `.notna()` checks, exactly
    the point-in-time-membership discipline this module's docstring point 2
    already calls for. Only an outright DUPLICATE (permno, date) row is
    still treated as a genuine integrity problem worth stopping the
    pipeline for -- that can never be an ordinary liquidity gap.
    """
    prices = pd.read_csv(prices_path, parse_dates=['dlycaldt'])

    dupes = prices.duplicated(subset=['permno', 'dlycaldt']).sum()
    assert dupes == 0, f"{dupes} duplicate (permno, date) rows in constituent price pull"

    prices = prices.copy()
    n_total = len(prices)
    n_price_nan = int(prices['dlyprc'].isna().sum())
    n_price_neg = int((prices['dlyprc'] < 0).sum())
    n_price_zero = int((prices['dlyprc'] == 0).sum())
    n_cumfac_bad = int((prices['dlycumfacpr'].isna() | (prices['dlycumfacpr'] <= 0)).sum())
    print(f"[price/cumfac quality] out of {n_total} rows: dlyprc NaN={n_price_nan} "
          f"({n_price_nan / n_total:.4%}), negative/bid-ask-average={n_price_neg} "
          f"({n_price_neg / n_total:.4%}, recovered via abs()), zero={n_price_zero} "
          f"({n_price_zero / n_total:.4%}, treated as invalid); dlycumfacpr NaN/non-positive="
          f"{n_cumfac_bad} ({n_cumfac_bad / n_total:.4%}, treated as invalid) -- these should "
          f"all be small fractions; a value far above ~1-2% is worth a closer look at the pull "
          f"itself, not assumed benign.")

    prices.loc[prices['dlyprc'] == 0, 'dlyprc'] = np.nan
    prices['dlyprc'] = prices['dlyprc'].abs()
    prices.loc[prices['dlycumfacpr'].isna() | (prices['dlycumfacpr'] <= 0), 'dlycumfacpr'] = np.nan

    prices['split_adj_price'] = prices['dlyprc'] / prices['dlycumfacpr']
    return prices


def build_price_panel(prices):
    """Wide date x permno panel of split-adjusted price."""
    panel = prices.pivot(index='dlycaldt', columns='permno', values='split_adj_price').sort_index()
    panel.index.name = 'date'
    return panel


def compute_above_200dma_panel(price_panel, ma_window=MA_WINDOW):
    """
    For each permno column: NaN before `ma_window` valid observations
    exist (that name's own price history isn't long enough yet to define
    its 200-day MA); 1.0/0.0 (above/at-or-below its own trailing MA)
    once defined.
    """
    rolling_ma = price_panel.rolling(window=ma_window, min_periods=ma_window).mean()
    above = pd.DataFrame(np.nan, index=price_panel.index, columns=price_panel.columns)
    valid = rolling_ma.notna() & price_panel.notna()
    above[valid] = (price_panel[valid] >= rolling_ma[valid]).astype(float)
    return above


def build_membership_mask(membership, dates, last_price_date=None):
    """
    membership : DataFrame with columns permno, start, ending (ending may
        be NaT for an open-ended/still-active spell).
    dates : DatetimeIndex to build the mask over (the price panel's own
        date index).
    last_price_date : the last date actually covered by the price pull --
        an open-ended `ending` is treated as extending through this date,
        not literally forever. Defaults to `dates.max()`.

    Returns
    -------
    mask : DataFrame, date x permno, boolean -- True where that permno was
        an actual S&P 500 member on that date.
    """
    if last_price_date is None:
        last_price_date = dates.max()
    membership = membership.copy()
    membership['ending'] = membership['ending'].fillna(last_price_date)

    permnos = sorted(membership['permno'].unique())
    mask = pd.DataFrame(False, index=dates, columns=permnos)
    for _, row in membership.iterrows():
        in_window = (dates >= row['start']) & (dates <= row['ending'])
        mask.loc[in_window, row['permno']] = True
    return mask


def compute_breadth_series(above_200dma_panel, membership_mask):
    """
    Combines the two panels: a permno counts on a date only if it is BOTH
    a genuine member that date AND has a defined 200-day MA that date.

    Returns
    -------
    breadth_pct : pd.Series, date-indexed, fraction (0-1) of eligible
        members trading above their own 200-day MA. NaN on any date with
        zero eligible members (should not occur in practice once the
        panel is past its earliest warm-up dates, but handled explicitly
        rather than silently divided-by-zero).
    n_eligible : pd.Series, the (diagnostic) count of eligible members
        per date -- worth checking against the ~500 sanity expectation
        before trusting the resulting breadth series.
    """
    common_cols = above_200dma_panel.columns.intersection(membership_mask.columns)
    above_c = above_200dma_panel[common_cols]
    member_c = membership_mask[common_cols].reindex(above_200dma_panel.index, fill_value=False)

    eligible = member_c & above_c.notna()
    n_eligible = eligible.sum(axis=1)
    n_above = (eligible & (above_c == 1.0)).sum(axis=1)

    breadth_pct = (n_above / n_eligible.replace(0, np.nan))
    return breadth_pct, n_eligible


def run_breadth_data_prep(membership_path=MEMBERSHIP_PATH, prices_path=PRICES_PATH):
    membership = pd.read_csv(membership_path, parse_dates=['start', 'ending'])
    prices = load_and_validate_prices(prices_path)
    price_panel = build_price_panel(prices)

    above_200dma = compute_above_200dma_panel(price_panel)
    membership_mask = build_membership_mask(membership, price_panel.index)
    breadth_pct, n_eligible = compute_breadth_series(above_200dma, membership_mask)

    print(f"[check] eligible-member count per date: min={n_eligible.min():.0f}, "
          f"median={n_eligible.median():.0f}, max={n_eligible.max():.0f} "
          f"(expect roughly 450-520 once past the panel's own warm-up period; "
          f"a value far outside that range signals a membership or price-pull problem)")
    return breadth_pct, n_eligible


# === RUN_FROM_HERE ===

if __name__ == '__main__':
    import os
    if not os.path.exists(MEMBERSHIP_PATH) or not os.path.exists(PRICES_PATH):
        missing = [p for p in (MEMBERSHIP_PATH, PRICES_PATH) if not os.path.exists(p)]
        print(f"[WAITING ON DATA] missing: {missing}")
    else:
        breadth_pct, n_eligible = run_breadth_data_prep()
        os.makedirs('data/raw', exist_ok=True)
        breadth_pct.dropna().to_frame('pct_above_200dma').to_csv(
            'data/raw/sp500_breadth_pct_above_200dma.csv')
        n_eligible.to_frame('n_eligible_members').to_csv(
            'data/raw/sp500_breadth_n_eligible_members.csv')
        print(f"\nSaved: data/raw/sp500_breadth_pct_above_200dma.csv "
              f"({breadth_pct.dropna().index.min().date()} to {breadth_pct.dropna().index.max().date()}), "
              f"data/raw/sp500_breadth_n_eligible_members.csv")

[price/cumfac quality] out of 12132066 rows: dlyprc NaN=17729 (0.1461%), negative/bid-ask-average=0 (0.0000%, recovered via abs()), zero=561 (0.0046%, treated as invalid); dlycumfacpr NaN/non-positive=637 (0.0053%, treated as invalid) -- these should all be small fractions; a value far above ~1-2% is worth a closer look at the pull itself, not assumed benign.
[check] eligible-member count per date: min=0, median=199, max=505 (expect roughly 450-520 once past the panel's own warm-up period; a value far outside that range signals a membership or price-pull problem)

Saved: data/raw/sp500_breadth_pct_above_200dma.csv (1926-08-30 to 2024-12-31), data/raw/sp500_breadth_n_eligible_members.csv


In [ ]:
import os
os.rename('/content/data/raw/tri_full_panel_14assets .csv', '/content/data/raw/tri_full_panel_14assets.csv')

In [ ]:
import time
import numpy as np
import pandas as pd

import condition_features as cf
from trend_meanrev_signal import build_trend_filtered_signal
from condedge_bet_dataset import extract_trades_for_asset

SEED = 20260930
NREPS_DEFAULT = 2000  # this project's standing convention -- see module
                       # docstring point 6 for why the real run below uses less.

GROUPS = {'HIGH': 1.0, 'LOW': -1.0}


# ==========================================================================
# Price-path reconstruction and the two summary statistics -- verbatim
# logic from trend_meanrev_mcpt.py (duplicated per this project's own
# standing practice of keeping each phase-script's small building blocks
# self-contained, e.g. CombPurgedKFoldCV's own repeated duplication).
# ==========================================================================

def reconstruct_price(price_series, permuted_returns, valid_mask):
    out = np.full(len(price_series), np.nan)
    valid_idx = np.flatnonzero(valid_mask)
    p0 = price_series.to_numpy()[valid_idx[0]]
    permuted_path = p0 * np.cumprod(np.concatenate([[1.0], 1 + permuted_returns]))
    out[valid_idx] = permuted_path
    return pd.Series(out, index=price_series.index)


def ann_sharpe(ret):
    ret = np.asarray(ret, dtype=float)
    ret = ret[~np.isnan(ret)]
    if len(ret) == 0:
        return np.nan
    std = ret.std(ddof=1) if len(ret) > 1 else 0.0
    if std == 0 or np.isnan(std):
        return np.nan
    return (ret.mean() / std) * np.sqrt(252)


def cum_return(ret):
    ret = np.asarray(ret, dtype=float)
    ret = ret[~np.isnan(ret)]
    if len(ret) == 0:
        return np.nan
    return float(np.prod(1 + ret) - 1)


# ==========================================================================
# Group-restricted day-level P&L (the genuinely new piece for this step).
# ==========================================================================

def build_group_position(bets, executed_position, daily_index, condition, label_value):
    """
    Returns a Series, same index as daily_index/executed_position: the
    rule's own executed position on days inside a trade whose `condition`
    column equals `label_value`, 0.0 elsewhere. Trades for one asset never
    overlap (extract_trades_for_asset's own sequential-scan guarantee), so
    there is no double-counting risk.
    """
    mask = np.zeros(len(daily_index), dtype=bool)
    sub = bets[bets[condition] == label_value]
    for t0, t1 in zip(sub['t0'], sub['t1']):
        start = daily_index.get_loc(t0)
        end = daily_index.get_loc(t1)  # t1 = first FLAT day after the run -- exclusive
        mask[start:end] = True
    return executed_position.where(pd.Series(mask, index=daily_index), 0.0)


def group_stats_for_trial(asset, price_col_series, hyg_series, lqd_series, yields_df, daily_index):
    """
    One full trial for one asset: rebuild the signal, extract trades,
    attach all 5 condition labels, then compute (ann_sharpe, cum_return)
    for the HIGH and LOW group of EACH condition from the same trial's
    trades -- the expensive parts (signal, trades, SADF) are computed once
    and shared across all 5 conditions x 2 groups, not repeated per
    condition.

    Returns: dict {condition: {'HIGH': (sharpe, cumret), 'LOW': (sharpe, cumret)}}
    """
    trial_panel = pd.DataFrame({asset: price_col_series}, index=daily_index)
    if asset != 'HYG':
        trial_panel['HYG'] = hyg_series
    if asset != 'LQD':
        trial_panel['LQD'] = lqd_series

    executed_position = build_trend_filtered_signal(trial_panel[[asset]])[asset]
    simple_ret = trial_panel[asset].pct_change(fill_method=None)
    trades = extract_trades_for_asset(asset, executed_position, simple_ret, daily_index)
    bets = pd.DataFrame(trades)

    out = {}
    if len(bets) == 0:
        for cond in cf.ALL_CONDITIONS:
            out[cond] = {'HIGH': (np.nan, np.nan), 'LOW': (np.nan, np.nan)}
        return out

    bets = bets[bets['decision_date'].notna()].reset_index(drop=True)
    labeled = cf.label_trades_by_condition(bets, trial_panel, yields_df)

    trend_valid = trial_panel[asset].pct_change(cf.TREND_LOOKBACK, fill_method=None).notna()

    for cond in cf.ALL_CONDITIONS:
        cond_out = {}
        for group_name, label_value in GROUPS.items():
            group_pos = build_group_position(labeled, executed_position, daily_index, cond, label_value)
            group_pnl = (group_pos * simple_ret).where(trend_valid)
            cond_out[group_name] = (ann_sharpe(group_pnl.to_numpy()), cum_return(group_pnl.to_numpy()))
        out[cond] = cond_out
    return out


# ==========================================================================
# Full per-asset permutation test.
# ==========================================================================

def run_asset_condition_mcpt(price_panel, yields_df, asset, rng, nreps=NREPS_DEFAULT):
    daily_index = price_panel.index
    hyg_series = price_panel['HYG'] if 'HYG' in price_panel.columns else None
    lqd_series = price_panel['LQD'] if 'LQD' in price_panel.columns else None

    price_series = price_panel[asset]
    valid_mask = price_series.notna().to_numpy()
    valid_prices = price_series.to_numpy()[valid_mask]
    real_returns = valid_prices[1:] / valid_prices[:-1] - 1.0

    # Trial 0: real, unpermuted price -- must reproduce label_trades_by_condition's
    # own independent output exactly (checked in the test suite).
    real_stats = group_stats_for_trial(asset, price_series, hyg_series, lqd_series, yields_df, daily_index)

    counts = {cond: {g: {'sharpe': 1, 'cumret': 1} for g in GROUPS} for cond in cf.ALL_CONDITIONS}

    for _ in range(1, nreps):
        permuted_returns = rng.permutation(real_returns)
        permuted_price = reconstruct_price(price_series, permuted_returns, valid_mask)
        trial_stats = group_stats_for_trial(asset, permuted_price, hyg_series, lqd_series, yields_df, daily_index)
        for cond in cf.ALL_CONDITIONS:
            for g in GROUPS:
                t_sharpe, t_cumret = trial_stats[cond][g]
                r_sharpe, r_cumret = real_stats[cond][g]
                if not np.isnan(t_sharpe) and not np.isnan(r_sharpe) and t_sharpe >= r_sharpe:
                    counts[cond][g]['sharpe'] += 1
                if not np.isnan(t_cumret) and not np.isnan(r_cumret) and t_cumret >= r_cumret:
                    counts[cond][g]['cumret'] += 1

    rows = []
    for cond in cf.ALL_CONDITIONS:
        for g in GROUPS:
            r_sharpe, r_cumret = real_stats[cond][g]
            rows.append({
                'asset': asset, 'condition': cond, 'group': g,
                'real_ann_sharpe': r_sharpe, 'p_value_sharpe': counts[cond][g]['sharpe'] / nreps,
                'real_cum_return': r_cumret, 'p_value_cum_return': counts[cond][g]['cumret'] / nreps,
            })
    return pd.DataFrame(rows)


def run_full_condition_mcpt(price_panel, yields_df, assets=None, nreps=NREPS_DEFAULT, seed=SEED):
    if assets is None:
        assets = list(price_panel.columns)
    rng = np.random.default_rng(seed)
    frames = [run_asset_condition_mcpt(price_panel, yields_df, asset, rng, nreps=nreps) for asset in assets]
    return pd.concat(frames, ignore_index=True)


# === RUN_FROM_HERE ===

if __name__ == '__main__':
    import os
    import sys
    for _d in dict.fromkeys(
        [os.getcwd()] + ([os.path.dirname(os.path.abspath(__file__))] if '__file__' in dir() else [])
    ):
        if _d not in sys.path:
            sys.path.insert(0, _d)

    TRI_PATH = 'data/raw/tri_full_panel_14assets.csv'
    YIELD_PATH = 'data/raw/treasury_yields_fred.csv'

    price_panel = pd.read_csv(TRI_PATH, index_col=0, parse_dates=True)
    yields_df = pd.read_csv(YIELD_PATH, index_col=0, parse_dates=True)

    # Benchmark ONE full trial (all 5 conditions, both groups) on the most
    # expensive asset (longest history -> most trades -> most SADF calls)
    # before committing to a NREPS for the real run -- measured, not guessed.
    lengths = price_panel.notna().sum().sort_values(ascending=False)
    worst_asset = lengths.index[0]
    daily_index = price_panel.index
    t0 = time.time()
    _ = group_stats_for_trial(worst_asset, price_panel[worst_asset],
                               price_panel.get('HYG'), price_panel.get('LQD'),
                               yields_df, daily_index)
    per_trial_s = time.time() - t0
    print(f"Benchmark: one trial for {worst_asset} (longest series) took {per_trial_s:.3f}s")

    n_assets = len(price_panel.columns)
    budget_s = 480  # leave headroom under the 10-minute tool-call cap
    nreps = max(2, int(budget_s / (per_trial_s * n_assets)))
    print(f"Budget {budget_s}s / ({n_assets} assets x {per_trial_s:.3f}s/trial) "
          f"-> NREPS={nreps} (standing convention is {NREPS_DEFAULT}; reduced here "
          f"for the reason flagged in this module's own docstring, point 6)")

    t0 = time.time()
    results = run_full_condition_mcpt(price_panel, yields_df, nreps=nreps, seed=SEED)
    elapsed = time.time() - t0
    print(f"\nFull run completed in {elapsed:.1f}s ({nreps} trials x {n_assets} assets)")

    pd.set_option('display.width', 160)
    sig = results[(results['p_value_sharpe'] < 0.05) | (results['p_value_cum_return'] < 0.05)]
    print(f"\n{len(sig)} / {len(results)} (asset, condition, group) rows significant on "
          f"Sharpe or cum. return at nominal p<0.05 (UNCORRECTED -- Step 5's DSR-style "
          f"pooled correction across all {len(results)} tests is required before any "
          f"of this is treated as real):")
    print(sig.sort_values('p_value_sharpe').to_string(index=False))

    os.makedirs('data/raw', exist_ok=True)
    results.to_csv('data/raw/condedge_pricepath_mcpt_results_14assets.csv', index=False)
    print(f"\nSaved: data/raw/condedge_pricepath_mcpt_results_14assets.csv "
          f"(NREPS={nreps}, seed={SEED})")

Benchmark: one trial for SPY (longest series) took 1.354s
Budget 480s / (14 assets x 1.354s/trial) -> NREPS=25 (standing convention is 2000; reduced here for the reason flagged in this module's own docstring, point 6)

Full run completed in 243.2s (25 trials x 14 assets)

15 / 140 (asset, condition, group) rows significant on Sharpe or cum. return at nominal p<0.05 (UNCORRECTED -- Step 5's DSR-style pooled correction across all 140 tests is required before any of this is treated as real):
asset        condition group  real_ann_sharpe  p_value_sharpe  real_cum_return  p_value_cum_return
  EEM       vol_regime  HIGH         0.343473            0.04         0.691940                0.04
  EEM   trend_strength   LOW         0.385449            0.04         0.796662                0.04
  EEM structural_break  HIGH         0.623021            0.04         1.391544                0.04
  HYG       vol_regime   LOW         0.492584            0.04         0.216609                0.08
  HYG   tre

In [ ]:
import shutil
shutil.move('/content/data/raw/condition_features.py', '/content/condition_features.py')

'/content/condition_features.py'

In [ ]:
import shutil
shutil.move('/content/data/raw/condedge_pricepath_mcpt.py', '/content/condedge_pricepath_mcpt.py')

'/content/condedge_pricepath_mcpt.py'

In [ ]:
import time
import numpy as np
import pandas as pd

import condition_features as cf
from trend_meanrev_signal import build_trend_filtered_signal
from condedge_bet_dataset import extract_trades_for_asset
from condedge_pricepath_mcpt import (
    reconstruct_price, ann_sharpe, cum_return, build_group_position,
    group_stats_for_trial, run_asset_condition_mcpt, run_full_condition_mcpt,
)

ok = True
def check(name, cond):
    global ok
    print(f"[{'PASS' if cond else 'FAIL'}] {name}")
    if not cond:
        ok = False

# ==========================================================================
# 1) MECHANICS: reconstruct_price -- identity permutation reproduces the
# original path exactly; a genuine shuffle preserves the exact return
# multiset while changing the path. (Same checks trend_meanrev_mcpt.py's
# own test suite established for this identical function.)
# ==========================================================================
dates = pd.bdate_range('2000-01-03', periods=500)
rng0 = np.random.default_rng(0)
ret0 = rng0.normal(0.0004, 0.012, len(dates))
price0 = pd.Series(100 * np.cumprod(1 + ret0), index=dates)
valid_mask0 = price0.notna().to_numpy()

identity_recon = reconstruct_price(price0, ret0[1:], valid_mask0)
# NOTE: ret0[1:] are the "returns from day1 onward" -- reconstruct_price's
# own contract takes valid_mask.sum()-1 returns and re-anchors at the same
# p0, so feeding it price0's OWN un-permuted post-inception returns must
# reproduce price0 exactly.
manual_returns = price0.to_numpy()[1:] / price0.to_numpy()[:-1] - 1.0
identity_recon2 = reconstruct_price(price0, manual_returns, valid_mask0)
check("MECHANICS: reconstruct_price with the asset's OWN real returns "
      "reproduces the original price path exactly",
      np.allclose(identity_recon2.to_numpy(), price0.to_numpy(), atol=1e-9))

shuffled = np.random.default_rng(5).permutation(manual_returns)
shuffled_recon = reconstruct_price(price0, shuffled, valid_mask0)
check("MECHANICS: a shuffled-returns reconstruction changes the path",
      not np.allclose(shuffled_recon.to_numpy(), price0.to_numpy(), atol=1e-6))
recovered_returns = shuffled_recon.to_numpy()[1:] / shuffled_recon.to_numpy()[:-1] - 1.0
check("MECHANICS: the shuffled reconstruction's OWN implied returns are "
      "exactly the same multiset as the shuffled input (just recompounded)",
      np.allclose(sorted(recovered_returns), sorted(shuffled), atol=1e-9))

# ==========================================================================
# 2) BUILD_GROUP_POSITION -- fully hand-checked on a small synthetic
# 2-trade example.
# ==========================================================================
idx = pd.bdate_range('2010-01-04', periods=10)
exec_pos = pd.Series([0, 1, 1, 1, 0, -1, -1, 0, 0, 0], index=idx, dtype=float)
bets_gp = pd.DataFrame({
    't0': [idx[1], idx[5]], 't1': [idx[4], idx[7]],
    'cond_x': [1.0, -1.0],   # trade 1 = HIGH, trade 2 = LOW
})
high_pos = build_group_position(bets_gp, exec_pos, idx, 'cond_x', 1.0)
low_pos = build_group_position(bets_gp, exec_pos, idx, 'cond_x', -1.0)

expected_high = np.array([0, 1, 1, 1, 0, 0, 0, 0, 0, 0], dtype=float)
expected_low = np.array([0, 0, 0, 0, 0, -1, -1, 0, 0, 0], dtype=float)
check("BUILD_GROUP_POSITION: HIGH-group position exactly isolates trade 1's "
      "own [t0, t1) window and zeroes everything else",
      np.array_equal(high_pos.to_numpy(), expected_high))
check("BUILD_GROUP_POSITION: LOW-group position exactly isolates trade 2's "
      "own [t0, t1) window and zeroes everything else",
      np.array_equal(low_pos.to_numpy(), expected_low))
check("BUILD_GROUP_POSITION: the two groups partition the trades with no overlap",
      not np.any((high_pos.to_numpy() != 0) & (low_pos.to_numpy() != 0)))

# ==========================================================================
# 3) TRIAL-0 CONSISTENCY: group_stats_for_trial on the REAL (unpermuted)
# price must reproduce an INDEPENDENT direct computation (build signal ->
# extract trades -> label -> restrict -> Sharpe) exactly.
# ==========================================================================
n = 1600
rng2 = np.random.default_rng(11)


def make_meanrevertible_price(n, seed, mr_strength=0.03, drift=0.0002, vol=0.01):
    r = np.random.default_rng(seed).normal(0, vol, n)
    log_p = np.zeros(n)
    log_p[0] = np.log(100.0)
    level = np.log(100.0)
    for i in range(1, n):
        level += drift
        log_p[i] = log_p[i - 1] + r[i] - mr_strength * (log_p[i - 1] - level)
    return np.exp(log_p)


e_dates = pd.bdate_range('2000-01-03', periods=n)
asset_price = make_meanrevertible_price(n, seed=1)
hyg_price = 50 * np.cumprod(1 + rng2.normal(0.0002, 0.005, n))
lqd_price = 80 * np.cumprod(1 + rng2.normal(0.0001, 0.004, n))
e_panel = pd.DataFrame({'A1': asset_price, 'HYG': hyg_price, 'LQD': lqd_price}, index=e_dates)
e_yields = pd.DataFrame({
    'DGS10': 3.0 + 0.5 * np.sin(np.linspace(0, 6, n)),
    'DGS3MO': 2.0 + 0.5 * np.cos(np.linspace(0, 6, n)),
}, index=e_dates)

t0 = time.time()
trial0_stats = group_stats_for_trial('A1', e_panel['A1'], e_panel['HYG'], e_panel['LQD'], e_yields, e_dates)
print(f"[timing] one trial (n={n}) took {time.time() - t0:.3f}s")

# Independent direct computation:
exec_pos_direct = build_trend_filtered_signal(e_panel[['A1']])['A1']
simple_ret_direct = e_panel['A1'].pct_change(fill_method=None)
trades_direct = extract_trades_for_asset('A1', exec_pos_direct, simple_ret_direct, e_dates)
bets_direct = pd.DataFrame(trades_direct)
bets_direct = bets_direct[bets_direct['decision_date'].notna()].reset_index(drop=True)
labeled_direct = cf.label_trades_by_condition(bets_direct, e_panel, e_yields)
trend_valid_direct = e_panel['A1'].pct_change(cf.TREND_LOOKBACK, fill_method=None).notna()

mismatches = []
for cond in cf.ALL_CONDITIONS:
    for group_name, label_value in [('HIGH', 1.0), ('LOW', -1.0)]:
        gp = build_group_position(labeled_direct, exec_pos_direct, e_dates, cond, label_value)
        gpnl = (gp * simple_ret_direct).where(trend_valid_direct)
        exp_sharpe, exp_cumret = ann_sharpe(gpnl.to_numpy()), cum_return(gpnl.to_numpy())
        got_sharpe, got_cumret = trial0_stats[cond][group_name]
        same = (np.isnan(exp_sharpe) and np.isnan(got_sharpe)) or np.isclose(exp_sharpe, got_sharpe, equal_nan=True)
        same = same and ((np.isnan(exp_cumret) and np.isnan(got_cumret)) or np.isclose(exp_cumret, got_cumret, equal_nan=True))
        if not same:
            mismatches.append((cond, group_name, exp_sharpe, got_sharpe, exp_cumret, got_cumret))
check("TRIAL-0 CONSISTENCY: group_stats_for_trial on real data matches an "
      "independently-computed signal->trades->label->restrict->Sharpe chain "
      "exactly, for all 5 conditions x 2 groups",
      len(mismatches) == 0)
if mismatches:
    print("  mismatches:", mismatches)

# ==========================================================================
# 4) P-VALUE SANITY + REPRODUCIBILITY on the same small synthetic panel
# (nreps kept small for test runtime -- this is a mechanics check, not the
# real run's own statistical resolution).
# ==========================================================================
t0 = time.time()
res_a = run_asset_condition_mcpt(e_panel, e_yields, 'A1', np.random.default_rng(42), nreps=25)
print(f"[timing] 25-trial run took {time.time() - t0:.1f}s")

check("P-VALUE SANITY: all p-values lie in (0, 1]",
      res_a['p_value_sharpe'].between(0, 1, inclusive='right').all()
      and res_a['p_value_cum_return'].between(0, 1, inclusive='right').all())
check("P-VALUE SANITY: every (condition, group) row present (5 conditions x 2 groups = 10 rows)",
      len(res_a) == 10)

res_b_sameseed = run_asset_condition_mcpt(e_panel, e_yields, 'A1', np.random.default_rng(42), nreps=25)
check("REPRODUCIBILITY: same seed -> identical p-values",
      np.allclose(res_a['p_value_sharpe'].to_numpy(), res_b_sameseed['p_value_sharpe'].to_numpy()))

res_c_diffseed = run_asset_condition_mcpt(e_panel, e_yields, 'A1', np.random.default_rng(7), nreps=25)
check("REPRODUCIBILITY: a different seed produces at least one different p-value "
      "(the permutation draws actually matter)",
      not np.allclose(res_a['p_value_sharpe'].to_numpy(), res_c_diffseed['p_value_sharpe'].to_numpy()))

# ==========================================================================
# 5) HYG/LQD SELF-REFERENCE CHECK: when the TESTED asset is HYG itself,
# credit_spread must be recomputed using the TRIAL's own (permuted) HYG
# leg, not the real one -- i.e. it should differ from testing a
# non-HYG/LQD asset's credit_spread (which stays pinned to the real
# HYG/LQD throughout).
# ==========================================================================
rng3 = np.random.default_rng(21)
hyg_own_price = make_meanrevertible_price(n, seed=2)
panel_hyg_test = pd.DataFrame({'HYG': hyg_own_price, 'LQD': lqd_price}, index=e_dates)

stats_real = group_stats_for_trial('HYG', panel_hyg_test['HYG'], panel_hyg_test['HYG'],
                                    panel_hyg_test['LQD'], e_yields, e_dates)
valid_mask_hyg = panel_hyg_test['HYG'].notna().to_numpy()
real_rets_hyg = panel_hyg_test['HYG'].to_numpy()[1:] / panel_hyg_test['HYG'].to_numpy()[:-1] - 1.0
permuted_hyg = reconstruct_price(panel_hyg_test['HYG'], rng3.permutation(real_rets_hyg), valid_mask_hyg)
stats_permuted = group_stats_for_trial('HYG', permuted_hyg, permuted_hyg,
                                        panel_hyg_test['LQD'], e_yields, e_dates)

# Rebuild each trial's OWN credit_spread series directly to confirm they
# actually differ (i.e. credit_spread used the TRIAL's own HYG leg, not a
# value pinned to the real HYG throughout).
cs_real = cf.compute_credit_spread_series(pd.DataFrame({'HYG': panel_hyg_test['HYG'], 'LQD': panel_hyg_test['LQD']}))
cs_permuted = cf.compute_credit_spread_series(pd.DataFrame({'HYG': permuted_hyg, 'LQD': panel_hyg_test['LQD']}))
check("HYG SELF-REFERENCE: permuting HYG's own price changes HYG's own "
      "credit_spread series (it is NOT pinned to the real HYG leg when "
      "HYG itself is the tested asset)",
      not np.allclose(cs_real.dropna().to_numpy(), cs_permuted.reindex(cs_real.dropna().index).to_numpy(),
                       equal_nan=True))

print("\n" + ("ALL CHECKS PASSED" if ok else "SOME CHECKS FAILED"))

[PASS] MECHANICS: reconstruct_price with the asset's OWN real returns reproduces the original price path exactly
[PASS] MECHANICS: a shuffled-returns reconstruction changes the path
[PASS] MECHANICS: the shuffled reconstruction's OWN implied returns are exactly the same multiset as the shuffled input (just recompounded)
[PASS] BUILD_GROUP_POSITION: HIGH-group position exactly isolates trade 1's own [t0, t1) window and zeroes everything else
[PASS] BUILD_GROUP_POSITION: LOW-group position exactly isolates trade 2's own [t0, t1) window and zeroes everything else
[PASS] BUILD_GROUP_POSITION: the two groups partition the trades with no overlap
[timing] one trial (n=1600) took 0.366s
[PASS] TRIAL-0 CONSISTENCY: group_stats_for_trial on real data matches an independently-computed signal->trades->label->restrict->Sharpe chain exactly, for all 5 conditions x 2 groups
[timing] 25-trial run took 4.9s
[PASS] P-VALUE SANITY: all p-values lie in (0, 1]
[PASS] P-VALUE SANITY: every (condition, group

In [ ]:
import numpy as np
import pandas as pd

TRI_PATH = 'data/raw/tri_full_panel_14assets.csv'
YIELD_PATH = 'data/raw/treasury_yields_fred.csv'
BREADTH_PATH = 'data/raw/sp500_breadth_pct_above_200dma.csv'  # Step 3: real market_breadth,
                                                               # from wrds_breadth_pull.py ->
                                                               # breadth_data_prep.py

VOL_SHORT_WINDOW = 21
VOL_LONG_WINDOW = 252
TREND_LOOKBACK = 252
CREDIT_WINDOW = 60           # user's pre-registered spec (Phase F used 63)
SADF_LOOKBACK = 63           # unchanged from step3b_metalabel_final_feature_round.py
SADF_MIN_SAMPLE = 20
SADF_LAGS = 1
MIN_WARMUP_DAYS = 252        # calendar-day-since-inception gate, all 6 conditions

# market_breadth is now a genuine 6th daily condition (Step 3) -- see
# compute_market_breadth_series/market_breadth_placeholder below for how it
# is wired (real series if available, else the original all-NaN placeholder,
# so every caller that predates Step 3 keeps working unchanged).
DAILY_CONDITIONS = ['vol_regime', 'trend_strength', 'credit_spread', 'rate_environment', 'market_breadth']
ALL_CONDITIONS = DAILY_CONDITIONS + ['structural_break']


# ==========================================================================
# The four DAILY conditions -- one per-asset DataFrame each, same shape as
# price_panel, NaN until each condition's own warmup window is satisfied.
# ==========================================================================

def compute_vol_regime_panel(price_panel):
    """Per-asset trailing 21d/252d realized-vol ratio (own returns)."""
    ret = price_panel.pct_change(fill_method=None)
    vol_short = ret.rolling(VOL_SHORT_WINDOW, min_periods=VOL_SHORT_WINDOW).std()
    vol_long = ret.rolling(VOL_LONG_WINDOW, min_periods=VOL_LONG_WINDOW).std()
    return vol_short / vol_long


def compute_trend_strength_panel(price_panel):
    """Per-asset |trailing 252-day return| (own price)."""
    return price_panel.pct_change(TREND_LOOKBACK, fill_method=None).abs()


def compute_credit_spread_series(price_panel):
    """Market-wide HYG-minus-LQD trailing 60-day return differential."""
    for col in ['HYG', 'LQD']:
        assert col in price_panel.columns, f"compute_credit_spread_series requires column {col}"
    hyg_ret = price_panel['HYG'].pct_change(CREDIT_WINDOW, fill_method=None)
    lqd_ret = price_panel['LQD'].pct_change(CREDIT_WINDOW, fill_method=None)
    return hyg_ret - lqd_ret


def compute_rate_environment_series(yields_df):
    """
    Market-wide DGS10-minus-DGS3MO LEVEL, on FRED's own calendar (no
    forward-fill -- exact-date lookup only, same no-ffill discipline as
    carry_signal.py's own DGS20/DGS3MO handling).
    """
    dgs10 = pd.to_numeric(yields_df['DGS10'], errors='coerce')
    dgs3mo = pd.to_numeric(yields_df['DGS3MO'], errors='coerce')
    return (dgs10 - dgs3mo).dropna()


def build_daily_condition_panels(price_panel, yields_df, breadth_series=None):
    """
    breadth_series : optional pd.Series (date-indexed), the real
        pct_above_200dma market-wide series (compute_market_breadth_series's
        output). When None (the default -- e.g. before Step 3's real WRDS
        pull is in hand), market_breadth falls back to the original all-NaN
        placeholder, so every pre-Step-3 caller keeps working unchanged.

    Returns dict: condition name -> DataFrame (same shape/columns as
    price_panel). The three market-wide conditions (credit_spread,
    rate_environment, market_breadth) are broadcast to every column, then
    masked to NaN outside each asset's own active window (its own
    first/last valid price date) so the per-asset warmup/median logic below
    "sees" each asset's own history length, not the panel's.
    """
    active_mask = price_panel.notna()
    # forward/backward-fill-free active window per asset: True from that
    # asset's own first valid date through its own last valid date.
    first_valid = price_panel.apply(lambda s: s.first_valid_index())
    last_valid = price_panel.apply(lambda s: s.last_valid_index())
    window_mask = pd.DataFrame(False, index=price_panel.index, columns=price_panel.columns)
    for col in price_panel.columns:
        if first_valid[col] is None:
            continue
        window_mask.loc[first_valid[col]:last_valid[col], col] = True

    vol_regime = compute_vol_regime_panel(price_panel).where(window_mask)
    trend_strength = compute_trend_strength_panel(price_panel).where(window_mask)

    credit_spread_1d = compute_credit_spread_series(price_panel)
    credit_spread = pd.DataFrame(
        {col: credit_spread_1d for col in price_panel.columns}, index=price_panel.index
    ).where(window_mask)

    rate_env_1d = compute_rate_environment_series(yields_df).reindex(price_panel.index)
    rate_environment = pd.DataFrame(
        {col: rate_env_1d for col in price_panel.columns}, index=price_panel.index
    ).where(window_mask)

    if breadth_series is not None:
        breadth_1d = breadth_series.reindex(price_panel.index)
        market_breadth = pd.DataFrame(
            {col: breadth_1d for col in price_panel.columns}, index=price_panel.index
        ).where(window_mask)
    else:
        market_breadth = market_breadth_placeholder(price_panel).where(window_mask)

    return {
        'vol_regime': vol_regime,
        'trend_strength': trend_strength,
        'credit_spread': credit_spread,
        'rate_environment': rate_environment,
        'market_breadth': market_breadth,
    }


# ==========================================================================
# Point-in-time expanding-window per-asset median split (the core
# machinery, shared by all conditions -- daily or sparse).
# ==========================================================================

def expanding_median_label(daily_panel, min_warmup_days=MIN_WARMUP_DAYS):
    """
    daily_panel : DataFrame, DatetimeIndex, one column per asset -- a
        DAILY condition series already masked to each asset's own active
        window (NaN outside it, e.g. build_daily_condition_panels' output).

    Returns a same-shaped DataFrame of labels in {+1.0 (HIGH), -1.0 (LOW),
    NaN}. For asset A at row position i within A's own active window
    (i=0 at A's own first valid date), the label uses the expanding
    median of A's own valid values at positions [0, i-1] (STRICTLY prior
    -- today's own value never contributes to today's own threshold), and
    is NaN until i >= min_warmup_days, i.e. at least one full year of A's
    own calendar history has elapsed since ITS OWN inception.
    """
    out = pd.DataFrame(np.nan, index=daily_panel.index, columns=daily_panel.columns)
    for col in daily_panel.columns:
        s = daily_panel[col]
        first = s.first_valid_index()
        if first is None:
            continue
        window = s.loc[first:]
        vals = window.to_numpy()
        n = len(vals)
        # expanding median of strictly-prior valid values, via shift(1)
        prior_median = pd.Series(vals).shift(1).expanding(min_periods=1).median().to_numpy()
        labels = np.full(n, np.nan)
        for i in range(n):
            if i < min_warmup_days:
                continue
            if np.isnan(vals[i]) or np.isnan(prior_median[i]):
                continue
            labels[i] = 1.0 if vals[i] >= prior_median[i] else -1.0
        out.loc[window.index, col] = labels
    return out


def expanding_median_label_sparse(sparse_values_by_asset, inception_by_asset,
                                   min_warmup_days=MIN_WARMUP_DAYS):
    """
    Sparse-history variant, for structural_break (see module docstring's
    flagged compute-cost simplification). sparse_values_by_asset: dict of
    asset -> Series (DatetimeIndex = the sparse set of dates the condition
    was actually evaluated at for that asset, e.g. trade decision dates,
    sorted ascending). inception_by_asset: dict of asset -> that asset's
    own TRUE first valid price date (used for the calendar-day warmup gate
    -- NOT the first sparse-evaluation date, which is typically much later
    than inception).

    Returns dict: asset -> Series of labels in {+1.0, -1.0, NaN}, same
    index as that asset's own sparse_values series.
    """
    out = {}
    for asset, s in sparse_values_by_asset.items():
        s = s.sort_index()
        inception = inception_by_asset[asset]
        vals = s.to_numpy()
        n = len(vals)
        prior_median = pd.Series(vals).shift(1).expanding(min_periods=1).median().to_numpy()
        labels = np.full(n, np.nan)
        for i in range(n):
            days_since_inception = (s.index[i] - inception).days
            if days_since_inception < min_warmup_days:
                continue
            if np.isnan(vals[i]) or np.isnan(prior_median[i]):
                continue
            labels[i] = 1.0 if vals[i] >= prior_median[i] else -1.0
        out[asset] = pd.Series(labels, index=s.index)
    return out


# ==========================================================================
# Structural break (SADF) -- sparse, reused verbatim from
# step3b_metalabel_final_feature_round.py (unchanged math).
# ==========================================================================

def get_beta_parts(X, y):
    xy = X.T @ y
    xx = X.T @ X
    xx_inv = np.linalg.pinv(xx)
    return xx, xx_inv, xy


def get_betas(X, y):
    xx, xx_inv, xy = get_beta_parts(X, y)
    beta = xx_inv @ xy
    err = y - X @ beta
    beta_var = err.T @ err / (X.shape[0] - X.shape[1]) * xx_inv
    beta_std = np.sqrt(np.diag(beta_var))
    return beta, beta_std


def lag_x(x, lags):
    if isinstance(lags, int):
        lags = range(lags + 1)
    return np.concatenate([x[(np.max(lags) - lag):(len(x) - lag)].reshape(-1, 1) for lag in lags], axis=1)


def np_get_xy(log_price, regression, lags):
    log_rtn = log_price[1:] - log_price[:-1]
    X = lag_x(log_rtn, lags)
    n_skip = log_price.shape[0] - X.shape[0]
    y = log_rtn[n_skip - 1:]
    X[:, 0] = log_price[n_skip - 1:-1]
    if regression != 'n':
        X = np.append(X, np.ones((X.shape[0], 1)), axis=1)
    if regression[:2] == 'ct':
        trend = np.arange(X.shape[0]).reshape(-1, 1)
        X = np.append(X, trend, axis=1)
    if regression == 'ctt':
        X = np.append(X, trend ** 2, axis=1)
    return X, y


def get_bsadf(log_price, min_sample, regression='c', lags=3):
    log_price = np.asarray(log_price)
    X, y = np_get_xy(log_price, regression=regression, lags=lags)
    min_sample = max([int(2 * np.max(lags) + 9), min_sample])
    start_points = range(0, y.shape[0] - min_sample + 1)
    adf_vals = np.zeros(len(start_points))
    for start in start_points:
        X_sub, y_sub = X[start:], y[start:]
        beta, beta_error = get_betas(X_sub, y_sub)
        adf_vals[start] = beta[0] / beta_error[0]
    return np.nanquantile(adf_vals, 0.99)


def compute_structural_break_at_dates(price_series, dates, lookback=SADF_LOOKBACK,
                                       min_sample=SADF_MIN_SAMPLE, lags=SADF_LAGS):
    """
    price_series : Series, DatetimeIndex, one asset's own TRI level.
    dates : iterable of Timestamps to evaluate at (must be in price_series's
        own index; each needs `lookback` trailing valid observations).

    Returns a Series indexed by `dates` (dates without enough trailing
    history, or not present in price_series, are NaN).
    """
    log_price = np.log(price_series)
    out = {}
    for d in dates:
        if d not in log_price.index:
            out[d] = np.nan
            continue
        pos = log_price.index.get_loc(d)
        if pos + 1 < lookback:
            out[d] = np.nan
            continue
        window = log_price.iloc[pos - lookback + 1: pos + 1].to_numpy()
        if np.isnan(window).any():
            out[d] = np.nan
            continue
        out[d] = get_bsadf(window, min_sample=min_sample, regression='c', lags=lags)
    return pd.Series(out)


# ==========================================================================
# Market breadth -- Step 3: now WIRED to real data (wrds_breadth_pull.py ->
# breadth_data_prep.py's pct_above_200dma series), via
# compute_market_breadth_series below. market_breadth_placeholder is KEPT
# (not deleted) as the explicit fallback build_daily_condition_panels uses
# when no breadth_series is passed in, so every pre-Step-3 caller/test that
# doesn't supply one keeps its original all-NaN behavior unchanged.
# ==========================================================================

def compute_market_breadth_series(breadth_path=BREADTH_PATH):
    """
    Market-wide S&P 500 breadth: fraction of eligible constituents trading
    above their own 200-day MA on each date (breadth_data_prep.py's
    `pct_above_200dma`, built from wrds_breadth_pull.py's real CRSP pull --
    point-in-time membership, split-adjusted prices, no survivorship bias).

    Exact-date lookup only, no forward-fill -- same no-ffill discipline as
    compute_rate_environment_series: a date this series doesn't cover
    (e.g. outside the S&P 500 sample window) is simply absent/NaN once
    reindexed onto price_panel's own dates, flagged rather than silently
    patched with a stale prior value.
    """
    breadth = pd.read_csv(breadth_path, index_col=0, parse_dates=True)
    return breadth['pct_above_200dma']


def market_breadth_placeholder(price_panel):
    """
    Returns an all-NaN DataFrame, same shape as price_panel -- the fallback
    build_daily_condition_panels uses when breadth_series=None (i.e. no real
    pct_above_200dma data has been supplied yet). Kept for backward
    compatibility with every pre-Step-3 call site/test.
    """
    return pd.DataFrame(np.nan, index=price_panel.index, columns=price_panel.columns)


# ==========================================================================
# Orchestration: attach all 5 available condition HIGH/LOW labels to a
# bets/trades DataFrame (condedge_bet_dataset.extract_trades_for_asset's
# own output schema: one row per completed round-trip trade, with `asset`
# and `decision_date` columns) at each trade's own decision_date.
# ==========================================================================

def label_trades_by_condition(bets, price_panel, yields_df, breadth_series=None):
    """
    bets : DataFrame with columns 'asset', 'decision_date' (and whatever
        else condedge_bet_dataset.build_bet_dataset-style trade extraction
        produced -- this function only reads those two columns).
    breadth_series : optional, see build_daily_condition_panels -- passed
        straight through unchanged.

    Returns a copy of `bets` with 6 new columns, one per condition
    (vol_regime, trend_strength, credit_spread, rate_environment,
    market_breadth, structural_break), each in {+1.0 (HIGH), -1.0 (LOW), NaN}.
    """
    bets = bets.copy()
    daily_panels = build_daily_condition_panels(price_panel, yields_df, breadth_series=breadth_series)
    daily_labels = {name: expanding_median_label(panel) for name, panel in daily_panels.items()}

    for name in DAILY_CONDITIONS:
        lbl_panel = daily_labels[name]
        vals = np.full(len(bets), np.nan)
        for i, (asset, d) in enumerate(zip(bets['asset'], bets['decision_date'])):
            if asset in lbl_panel.columns and d in lbl_panel.index:
                vals[i] = lbl_panel.at[d, asset]
        bets[name] = vals

    # structural_break: sparse, one asset at a time, only at that asset's
    # own set of decision dates actually present in bets.
    first_valid = price_panel.apply(lambda s: s.first_valid_index())
    sb_labels_all = pd.Series(np.nan, index=bets.index)
    for asset, rows in bets.groupby('asset').groups.items():
        if asset not in price_panel.columns or first_valid[asset] is None:
            continue
        dates_needed = sorted(set(bets.loc[rows, 'decision_date']))
        sadf_vals = compute_structural_break_at_dates(price_panel[asset], dates_needed)
        sadf_labels = expanding_median_label_sparse(
            {asset: sadf_vals}, {asset: first_valid[asset]})[asset]
        for i in rows:
            d = bets.at[i, 'decision_date']
            if d in sadf_labels.index:
                sb_labels_all.at[i] = sadf_labels.at[d]
    bets['structural_break'] = sb_labels_all.values

    return bets


# === RUN_FROM_HERE ===

if __name__ == '__main__':
    import os
    import sys
    # Robust to BOTH execution styles: `!python condition_features.py` (where
    # __file__ is this script's own real path) AND pasting/running this file
    # as a Colab/Jupyter cell (where some kernels set __file__ to a USELESS
    # transient path like /tmp/ipykernel_.../<cell>.py, which silently sends
    # the old single-branch version of this line looking in the wrong
    # directory for sibling modules -- a real failure mode hit in practice,
    # not a hypothetical one). Try cwd FIRST (correct for a Colab cell,
    # where uploaded files land in the working directory), then __file__'s
    # own directory as a fallback (correct for a real script invocation).
    for _d in dict.fromkeys(
        [os.getcwd()] + ([os.path.dirname(os.path.abspath(__file__))] if '__file__' in dir() else [])
    ):
        if _d not in sys.path:
            sys.path.insert(0, _d)
    from trend_meanrev_signal import build_trend_filtered_signal
    from condedge_bet_dataset import extract_trades_for_asset

    price_panel = pd.read_csv(TRI_PATH, index_col=0, parse_dates=True)
    yields_df = pd.read_csv(YIELD_PATH, index_col=0, parse_dates=True)

    if os.path.exists(BREADTH_PATH):
        breadth_series = compute_market_breadth_series(BREADTH_PATH)
        print(f"Loaded real market_breadth series ({BREADTH_PATH}): "
              f"{breadth_series.index.min().date()} to {breadth_series.index.max().date()}, "
              f"{breadth_series.notna().sum()} valid observations")
    else:
        breadth_series = None
        print(f"[market_breadth STILL PLACEHOLDER] {BREADTH_PATH} not found -- "
              f"run wrds_breadth_pull.py then breadth_data_prep.py first. "
              f"Proceeding with market_breadth as all-NaN for now (5 usable conditions).")

    executed_position_panel = build_trend_filtered_signal(price_panel)
    daily_index = price_panel.index
    simple_ret_panel = price_panel.pct_change(fill_method=None)

    all_rows = []
    for asset in price_panel.columns:
        all_rows.extend(extract_trades_for_asset(
            asset, executed_position_panel[asset], simple_ret_panel[asset], daily_index))
    bets = pd.DataFrame(all_rows)
    bets = bets[bets['decision_date'].notna()].sort_values('t0').reset_index(drop=True)
    print(f"Total completed round-trip trades (14-asset universe): {len(bets)}")

    import time
    t0 = time.time()
    labeled = label_trades_by_condition(bets, price_panel, yields_df, breadth_series=breadth_series)
    print(f"Condition labeling completed in {time.time() - t0:.1f}s")

    pd.set_option('display.width', 160)
    print("\n--- HIGH/LOW group sizes per condition (NaN = warmup not cleared / missing input) ---")
    for cond in ALL_CONDITIONS:
        counts = labeled[cond].value_counts(dropna=False)
        n_high = counts.get(1.0, 0)
        n_low = counts.get(-1.0, 0)
        n_nan = counts.get(np.nan, 0) if labeled[cond].isna().any() else 0
        print(f"  {cond:18s}  HIGH={n_high:5d}  LOW={n_low:5d}  NaN={n_nan:5d}")

    os.makedirs('data/raw', exist_ok=True)
    labeled.to_csv('data/raw/condedge_trades_labeled_6cond_14assets.csv', index=False)
    breadth_note = "real data" if breadth_series is not None else "STILL PLACEHOLDER (all-NaN)"
    print(f"\nSaved: data/raw/condedge_trades_labeled_6cond_14assets.csv "
          f"(market_breadth column: {breadth_note})")

Loaded real market_breadth series (data/raw/sp500_breadth_pct_above_200dma.csv): 1926-08-30 to 2024-12-31, 25852 valid observations
Total completed round-trip trades (14-asset universe): 1309
Condition labeling completed in 7.1s

--- HIGH/LOW group sizes per condition (NaN = warmup not cleared / missing input) ---
  vol_regime          HIGH=  629  LOW=  635  NaN=   45
  trend_strength      HIGH=  431  LOW=  833  NaN=   45
  credit_spread       HIGH=  528  LOW=  515  NaN=  266
  rate_environment    HIGH=  397  LOW=  899  NaN=   13
  market_breadth      HIGH=  619  LOW=  657  NaN=   33
  structural_break    HIGH=  613  LOW=  682  NaN=   14

Saved: data/raw/condedge_trades_labeled_6cond_14assets.csv (market_breadth column: real data)


In [ ]:
import time
import numpy as np
import pandas as pd

import condition_features as cf
from trend_meanrev_signal import build_trend_filtered_signal
from condedge_bet_dataset import extract_trades_for_asset

SEED = 20260930
NREPS_DEFAULT = 2000  # this project's standing convention -- see module
                       # docstring point 6 for why the real run below uses less.
NREPS_REAL_RUN = 30    # FIXED, not re-derived from a live speed benchmark --
                        # see module docstring point 6a for why a benchmark-
                        # chosen NREPS was a real reproducibility bug (a
                        # slower machine silently gets a different, non-
                        # comparable trial count and therefore a different
                        # nominal result, even with the identical seed).
                        # Chosen conservatively below the slowest of two
                        # observed real benchmarks (0.909s/trial locally,
                        # 1.007s/trial on the user's own Colab instance) so
                        # it comfortably fits the runtime budget on either.

GROUPS = {'HIGH': 1.0, 'LOW': -1.0}


# ==========================================================================
# Price-path reconstruction and the two summary statistics -- verbatim
# logic from trend_meanrev_mcpt.py (duplicated per this project's own
# standing practice of keeping each phase-script's small building blocks
# self-contained, e.g. CombPurgedKFoldCV's own repeated duplication).
# ==========================================================================

def reconstruct_price(price_series, permuted_returns, valid_mask):
    out = np.full(len(price_series), np.nan)
    valid_idx = np.flatnonzero(valid_mask)
    p0 = price_series.to_numpy()[valid_idx[0]]
    permuted_path = p0 * np.cumprod(np.concatenate([[1.0], 1 + permuted_returns]))
    out[valid_idx] = permuted_path
    return pd.Series(out, index=price_series.index)


def ann_sharpe(ret):
    ret = np.asarray(ret, dtype=float)
    ret = ret[~np.isnan(ret)]
    if len(ret) == 0:
        return np.nan
    std = ret.std(ddof=1) if len(ret) > 1 else 0.0
    if std == 0 or np.isnan(std):
        return np.nan
    return (ret.mean() / std) * np.sqrt(252)


def cum_return(ret):
    ret = np.asarray(ret, dtype=float)
    ret = ret[~np.isnan(ret)]
    if len(ret) == 0:
        return np.nan
    return float(np.prod(1 + ret) - 1)


# ==========================================================================
# Group-restricted day-level P&L (the genuinely new piece for this step).
# ==========================================================================

def build_group_position(bets, executed_position, daily_index, condition, label_value):
    """
    Returns a Series, same index as daily_index/executed_position: the
    rule's own executed position on days inside a trade whose `condition`
    column equals `label_value`, 0.0 elsewhere. Trades for one asset never
    overlap (extract_trades_for_asset's own sequential-scan guarantee), so
    there is no double-counting risk.
    """
    mask = np.zeros(len(daily_index), dtype=bool)
    sub = bets[bets[condition] == label_value]
    for t0, t1 in zip(sub['t0'], sub['t1']):
        start = daily_index.get_loc(t0)
        end = daily_index.get_loc(t1)  # t1 = first FLAT day after the run -- exclusive
        mask[start:end] = True
    return executed_position.where(pd.Series(mask, index=daily_index), 0.0)


def group_stats_for_trial(asset, price_col_series, hyg_series, lqd_series, yields_df, daily_index,
                           breadth_series=None):
    """
    One full trial for one asset: rebuild the signal, extract trades,
    attach all 6 condition labels, then compute (ann_sharpe, cum_return)
    for the HIGH and LOW group of EACH condition from the same trial's
    trades -- the expensive parts (signal, trades, SADF) are computed once
    and shared across all 6 conditions x 2 groups, not repeated per
    condition.

    breadth_series : the real (or None) market_breadth series -- passed
        straight through to label_trades_by_condition, UNCHANGED across
        every trial (it is a genuinely external, market-wide S&P 500
        series with no dependence on any of the tested universe's own
        price columns, so it is never permuted -- same treatment as
        yields_df, which also stays real/constant across trials).

    Returns: dict {condition: {'HIGH': (sharpe, cumret), 'LOW': (sharpe, cumret)}}
    """
    trial_panel = pd.DataFrame({asset: price_col_series}, index=daily_index)
    if asset != 'HYG':
        trial_panel['HYG'] = hyg_series
    if asset != 'LQD':
        trial_panel['LQD'] = lqd_series

    executed_position = build_trend_filtered_signal(trial_panel[[asset]])[asset]
    simple_ret = trial_panel[asset].pct_change(fill_method=None)
    trades = extract_trades_for_asset(asset, executed_position, simple_ret, daily_index)
    bets = pd.DataFrame(trades)

    out = {}
    if len(bets) == 0:
        for cond in cf.ALL_CONDITIONS:
            out[cond] = {'HIGH': (np.nan, np.nan), 'LOW': (np.nan, np.nan)}
        return out

    bets = bets[bets['decision_date'].notna()].reset_index(drop=True)
    labeled = cf.label_trades_by_condition(bets, trial_panel, yields_df, breadth_series=breadth_series)

    trend_valid = trial_panel[asset].pct_change(cf.TREND_LOOKBACK, fill_method=None).notna()

    for cond in cf.ALL_CONDITIONS:
        cond_out = {}
        for group_name, label_value in GROUPS.items():
            group_pos = build_group_position(labeled, executed_position, daily_index, cond, label_value)
            group_pnl = (group_pos * simple_ret).where(trend_valid)
            cond_out[group_name] = (ann_sharpe(group_pnl.to_numpy()), cum_return(group_pnl.to_numpy()))
        out[cond] = cond_out
    return out


# ==========================================================================
# Full per-asset permutation test.
# ==========================================================================

def run_asset_condition_mcpt(price_panel, yields_df, asset, rng, nreps=NREPS_DEFAULT, breadth_series=None):
    daily_index = price_panel.index
    hyg_series = price_panel['HYG'] if 'HYG' in price_panel.columns else None
    lqd_series = price_panel['LQD'] if 'LQD' in price_panel.columns else None

    price_series = price_panel[asset]
    valid_mask = price_series.notna().to_numpy()
    valid_prices = price_series.to_numpy()[valid_mask]
    real_returns = valid_prices[1:] / valid_prices[:-1] - 1.0

    # Trial 0: real, unpermuted price -- must reproduce label_trades_by_condition's
    # own independent output exactly (checked in the test suite).
    real_stats = group_stats_for_trial(asset, price_series, hyg_series, lqd_series, yields_df, daily_index,
                                        breadth_series=breadth_series)

    counts = {cond: {g: {'sharpe': 1, 'cumret': 1} for g in GROUPS} for cond in cf.ALL_CONDITIONS}

    for _ in range(1, nreps):
        permuted_returns = rng.permutation(real_returns)
        permuted_price = reconstruct_price(price_series, permuted_returns, valid_mask)
        trial_stats = group_stats_for_trial(asset, permuted_price, hyg_series, lqd_series, yields_df, daily_index,
                                             breadth_series=breadth_series)
        for cond in cf.ALL_CONDITIONS:
            for g in GROUPS:
                t_sharpe, t_cumret = trial_stats[cond][g]
                r_sharpe, r_cumret = real_stats[cond][g]
                if not np.isnan(t_sharpe) and not np.isnan(r_sharpe) and t_sharpe >= r_sharpe:
                    counts[cond][g]['sharpe'] += 1
                if not np.isnan(t_cumret) and not np.isnan(r_cumret) and t_cumret >= r_cumret:
                    counts[cond][g]['cumret'] += 1

    rows = []
    for cond in cf.ALL_CONDITIONS:
        for g in GROUPS:
            r_sharpe, r_cumret = real_stats[cond][g]
            rows.append({
                'asset': asset, 'condition': cond, 'group': g,
                'real_ann_sharpe': r_sharpe, 'p_value_sharpe': counts[cond][g]['sharpe'] / nreps,
                'real_cum_return': r_cumret, 'p_value_cum_return': counts[cond][g]['cumret'] / nreps,
            })
    return pd.DataFrame(rows)


def run_full_condition_mcpt(price_panel, yields_df, assets=None, nreps=NREPS_DEFAULT, seed=SEED,
                             breadth_series=None):
    if assets is None:
        assets = list(price_panel.columns)
    rng = np.random.default_rng(seed)
    frames = [run_asset_condition_mcpt(price_panel, yields_df, asset, rng, nreps=nreps,
                                        breadth_series=breadth_series) for asset in assets]
    return pd.concat(frames, ignore_index=True)


# === RUN_FROM_HERE ===

if __name__ == '__main__':
    import os
    import sys
    for _d in dict.fromkeys(
        [os.getcwd()] + ([os.path.dirname(os.path.abspath(__file__))] if '__file__' in dir() else [])
    ):
        if _d not in sys.path:
            sys.path.insert(0, _d)

    TRI_PATH = 'data/raw/tri_full_panel_14assets.csv'
    YIELD_PATH = 'data/raw/treasury_yields_fred.csv'

    price_panel = pd.read_csv(TRI_PATH, index_col=0, parse_dates=True)
    yields_df = pd.read_csv(YIELD_PATH, index_col=0, parse_dates=True)

    if os.path.exists(cf.BREADTH_PATH):
        breadth_series = cf.compute_market_breadth_series(cf.BREADTH_PATH)
        print(f"Loaded real market_breadth series ({cf.BREADTH_PATH}): "
              f"{breadth_series.index.min().date()} to {breadth_series.index.max().date()}")
    else:
        breadth_series = None
        print(f"[market_breadth STILL PLACEHOLDER] {cf.BREADTH_PATH} not found -- "
              f"proceeding with market_breadth as all-NaN for now (5 usable conditions).")

    # Benchmark ONE full trial purely as an INFORMATIONAL runtime estimate --
    # NREPS itself is now a fixed constant (NREPS_REAL_RUN), never derived
    # from this live measurement. See module docstring point 6a: choosing
    # NREPS from a live per-machine benchmark was a real reproducibility
    # bug, caught by comparing two independent runs against each other.
    lengths = price_panel.notna().sum().sort_values(ascending=False)
    worst_asset = lengths.index[0]
    daily_index = price_panel.index
    t0 = time.time()
    _ = group_stats_for_trial(worst_asset, price_panel[worst_asset],
                               price_panel.get('HYG'), price_panel.get('LQD'),
                               yields_df, daily_index, breadth_series=breadth_series)
    per_trial_s = time.time() - t0
    n_assets = len(price_panel.columns)
    nreps = NREPS_REAL_RUN
    print(f"Benchmark (informational only): one trial for {worst_asset} (longest series) "
          f"took {per_trial_s:.3f}s -> full run estimated at ~{per_trial_s * n_assets * nreps:.0f}s")
    print(f"NREPS={nreps} (FIXED constant, standing convention is {NREPS_DEFAULT} -- reduced "
          f"here for the reason flagged in this module's own docstring, point 6/6a)")

    t0 = time.time()
    results = run_full_condition_mcpt(price_panel, yields_df, nreps=nreps, seed=SEED,
                                       breadth_series=breadth_series)
    elapsed = time.time() - t0
    print(f"\nFull run completed in {elapsed:.1f}s ({nreps} trials x {n_assets} assets)")

    pd.set_option('display.width', 160)
    sig = results[(results['p_value_sharpe'] < 0.05) | (results['p_value_cum_return'] < 0.05)]
    print(f"\n{len(sig)} / {len(results)} (asset, condition, group) rows significant on "
          f"Sharpe or cum. return at nominal p<0.05 (UNCORRECTED -- Step 5's DSR-style "
          f"pooled correction across all {len(results)} tests is required before any "
          f"of this is treated as real):")
    print(sig.sort_values('p_value_sharpe').to_string(index=False))

    os.makedirs('data/raw', exist_ok=True)
    results.to_csv('data/raw/condedge_pricepath_mcpt_results_14assets.csv', index=False)
    print(f"\nSaved: data/raw/condedge_pricepath_mcpt_results_14assets.csv "
          f"(NREPS={nreps}, seed={SEED})")

Loaded real market_breadth series (data/raw/sp500_breadth_pct_above_200dma.csv): 1926-08-30 to 2024-12-31
Benchmark (informational only): one trial for SPY (longest series) took 2.492s -> full run estimated at ~1046s
NREPS=30 (FIXED constant, standing convention is 2000 -- reduced here for the reason flagged in this module's own docstring, point 6/6a)

Full run completed in 351.1s (30 trials x 14 assets)

13 / 168 (asset, condition, group) rows significant on Sharpe or cum. return at nominal p<0.05 (UNCORRECTED -- Step 5's DSR-style pooled correction across all 168 tests is required before any of this is treated as real):
asset        condition group  real_ann_sharpe  p_value_sharpe  real_cum_return  p_value_cum_return
  EEM structural_break  HIGH         0.623021        0.033333         1.391544            0.033333
  HYG    credit_spread  HIGH         0.469835        0.033333         0.233863            0.033333
  QQQ   trend_strength   LOW         0.365932        0.033333         1.0

In [ ]:
!grep -n "BREADTH_PATH" condition_features.py

113:BREADTH_PATH = 'data/raw/sp500_breadth_pct_above_200dma.csv'  # Step 3: real market_breadth,
396:def compute_market_breadth_series(breadth_path=BREADTH_PATH):
499:    if os.path.exists(BREADTH_PATH):
500:        breadth_series = compute_market_breadth_series(BREADTH_PATH)
501:        print(f"Loaded real market_breadth series ({BREADTH_PATH}): "
506:        print(f"[market_breadth STILL PLACEHOLDER] {BREADTH_PATH} not found -- "


In [ ]:
import shutil
shutil.move('/content/data/raw/trend_meanrev_dsr.py', '/content/trend_meanrev_dsr.py')

'/content/trend_meanrev_dsr.py'

In [ ]:
import numpy as np
import pandas as pd

import condition_features as cf
from trend_meanrev_signal import build_trend_filtered_signal
from trend_meanrev_dsr import run_dsr
from condedge_pricepath_mcpt import build_group_position, GROUPS


def build_all_group_daily_returns(price_panel, labeled_bets):
    """
    labeled_bets : the Step 2 output (one row per completed trade, an
        'asset' column, and the 5 condition HIGH/LOW columns) for the REAL
        14-asset universe.

    Returns dict {"{asset}|{condition}|{group}": np.ndarray} -- the SAME
    group-restricted, tradeable-window-masked daily P&L series Step 4's
    MCPT used for its own real (trial-0) statistic, for all (asset,
    condition, group) combinations with at least one trade in that group.
    """
    daily_index = price_panel.index
    out = {}
    for asset, bets_asset in labeled_bets.groupby('asset'):
        if asset not in price_panel.columns:
            continue
        executed_position = build_trend_filtered_signal(price_panel[[asset]])[asset]
        simple_ret = price_panel[asset].pct_change(fill_method=None)
        trend_valid = price_panel[asset].pct_change(cf.TREND_LOOKBACK, fill_method=None).notna()

        for cond in cf.ALL_CONDITIONS:
            for group_name, label_value in GROUPS.items():
                gp = build_group_position(bets_asset, executed_position, daily_index, cond, label_value)
                gpnl = (gp * simple_ret).where(trend_valid).dropna()
                key = f"{asset}|{cond}|{group_name}"
                out[key] = gpnl.to_numpy()
    return out


# === RUN_FROM_HERE ===

if __name__ == '__main__':
    import os
    import sys
    for _d in dict.fromkeys(
        [os.getcwd()] + ([os.path.dirname(os.path.abspath(__file__))] if '__file__' in dir() else [])
    ):
        if _d not in sys.path:
            sys.path.insert(0, _d)

    TRI_PATH = 'data/raw/tri_full_panel_14assets.csv'
    LABELED_PATH = 'data/raw/condedge_trades_labeled_6cond_14assets.csv'
    MCPT_PATH = 'data/raw/condedge_pricepath_mcpt_results_14assets.csv'

    price_panel = pd.read_csv(TRI_PATH, index_col=0, parse_dates=True)
    labeled_bets = pd.read_csv(LABELED_PATH, parse_dates=['t0', 't1', 'decision_date'])
    mcpt_results = pd.read_csv(MCPT_PATH)

    daily_returns_by_trial = build_all_group_daily_returns(price_panel, labeled_bets)
    print(f"Built {len(daily_returns_by_trial)} group-restricted daily return series "
          f"(expect up to 140 = 14 assets x 5 conditions x 2 groups)")

    trial_inputs, summary = run_dsr(daily_returns_by_trial)
    trial_inputs[['asset', 'condition', 'group']] = trial_inputs['asset'].str.split('|', expand=True)
    summary[['asset', 'condition', 'group']] = summary['asset'].str.split('|', expand=True)

    pd.set_option('display.width', 160)
    n_survive_dsr = int(summary['survives_dsr_at_0.95'].sum())
    print(f"\nSR0 (daily, shared across all {len(summary)} trials): {summary['SR0_daily'].iloc[0]:.4f}")
    print(f"{n_survive_dsr} / {len(summary)} tests survive DSR alone (PSR >= 0.95 against SR0)")

    # Combine with Step 4's own MCPT verdict -- this project's standing
    # two-part rule: BOTH must be significant.
    combined = mcpt_results.merge(
        summary[['asset', 'condition', 'group', 'SR0_daily', 'sr_hat_daily', 'sr_hat_annualized', 'psr', 'survives_dsr_at_0.95']],
        on=['asset', 'condition', 'group'], how='left'
    )
    combined['survives_mcpt'] = (combined['p_value_sharpe'] < 0.05) | (combined['p_value_cum_return'] < 0.05)
    combined['survives_both'] = combined['survives_mcpt'] & combined['survives_dsr_at_0.95']

    n_survive_both = int(combined['survives_both'].sum())
    print(f"\n{n_survive_both} / {len(combined)} tests survive BOTH MCPT (nominal p<0.05) "
          f"AND DSR (PSR>=0.95) -- this project's standing two-part rule for accepting an edge as real.")
    if n_survive_both > 0:
        print(combined[combined['survives_both']].to_string(index=False))
    else:
        print("(none)")

    print(f"\nFor reference, MCPT-only survivors and their DSR fate:")
    mcpt_only = combined[combined['survives_mcpt']].sort_values('psr', ascending=False)
    print(mcpt_only[['asset', 'condition', 'group', 'p_value_sharpe', 'p_value_cum_return',
                      'sr_hat_annualized', 'psr', 'survives_dsr_at_0.95']].to_string(index=False))

    os.makedirs('data/raw', exist_ok=True)
    trial_inputs.to_csv('data/raw/condedge_dsr_trial_inputs.csv', index=False)
    combined.to_csv('data/raw/condedge_dsr_combined_verdict_14assets.csv', index=False)
    print("\nSaved: data/raw/condedge_dsr_trial_inputs.csv, data/raw/condedge_dsr_combined_verdict_14assets.csv")

Built 140 group-restricted daily return series (expect up to 140 = 14 assets x 5 conditions x 2 groups)

SR0 (daily, shared across all 140 trials): 0.0418
0 / 140 tests survive DSR alone (PSR >= 0.95 against SR0)

0 / 140 tests survive BOTH MCPT (nominal p<0.05) AND DSR (PSR>=0.95) -- this project's standing two-part rule for accepting an edge as real.
(none)

For reference, MCPT-only survivors and their DSR fate:
asset        condition group  p_value_sharpe  p_value_cum_return  sr_hat_annualized      psr  survives_dsr_at_0.95
  SPY       vol_regime   LOW            0.04                0.04           0.746916 0.686822                 False
  VNQ       vol_regime   LOW            0.04                0.04           0.670288 0.513722                 False
  SPY   trend_strength  HIGH            0.04                0.08           0.640039 0.447032                 False
  EEM structural_break  HIGH            0.04                0.04           0.623021 0.423535                 False
  SPY s

In [ ]:
import shutil
shutil.move('/content/data/raw/condedge_dsr.py', '/content/condedge_dsr.py')

'/content/condedge_dsr.py'

In [ ]:
import numpy as np
import pandas as pd

from trend_meanrev_signal import build_trend_filtered_signal
from condedge_bet_dataset import extract_trades_for_asset
import condition_features as cf
from condedge_dsr import build_all_group_daily_returns
from trend_meanrev_dsr import run_dsr, expected_max_sharpe, probabilistic_sharpe_ratio

ok = True
def check(name, cond):
    global ok
    print(f"[{'PASS' if cond else 'FAIL'}] {name}")
    if not cond:
        ok = False

# ==========================================================================
# 1) MECHANICS: build_all_group_daily_returns produces up to 10 series
# (5 conditions x 2 groups) per asset with a completed trade, each a plain
# array with no NaN, and a length matching the asset's own tradeable window.
# ==========================================================================
n = 1600
e_dates = pd.bdate_range('2000-01-03', periods=n)
rng = np.random.default_rng(3)


def make_meanrevertible_price(n, seed, mr_strength=0.03, drift=0.0002, vol=0.01):
    r = np.random.default_rng(seed).normal(0, vol, n)
    log_p = np.zeros(n)
    log_p[0] = np.log(100.0)
    level = np.log(100.0)
    for i in range(1, n):
        level += drift
        log_p[i] = log_p[i - 1] + r[i] - mr_strength * (log_p[i - 1] - level)
    return np.exp(log_p)


e_panel = pd.DataFrame({
    'A1': make_meanrevertible_price(n, seed=1),
    'A2': make_meanrevertible_price(n, seed=2),
    'HYG': 50 * np.cumprod(1 + rng.normal(0.0002, 0.005, n)),
    'LQD': 80 * np.cumprod(1 + rng.normal(0.0001, 0.004, n)),
}, index=e_dates)
e_yields = pd.DataFrame({
    'DGS10': 3.0 + 0.5 * np.sin(np.linspace(0, 6, n)),
    'DGS3MO': 2.0 + 0.5 * np.cos(np.linspace(0, 6, n)),
}, index=e_dates)

all_rows = []
for asset in ['A1', 'A2']:
    exec_pos = build_trend_filtered_signal(e_panel[[asset]])[asset]
    simple_ret = e_panel[asset].pct_change(fill_method=None)
    all_rows.extend(extract_trades_for_asset(asset, exec_pos, simple_ret, e_dates))
bets = pd.DataFrame(all_rows)
bets = bets[bets['decision_date'].notna()].sort_values('t0').reset_index(drop=True)
labeled = cf.label_trades_by_condition(bets, e_panel, e_yields)

series_dict = build_all_group_daily_returns(e_panel[['A1', 'A2', 'HYG', 'LQD']], labeled)
check("MECHANICS: at most 20 series (2 assets x 5 conditions x 2 groups)",
      len(series_dict) <= 20 and len(series_dict) > 0)
check("MECHANICS: every returned series is NaN-free",
      all(not np.isnan(v).any() for v in series_dict.values()))
check("MECHANICS: series keys are exactly 'asset|condition|group' with a "
      "known asset, a known condition, and a valid group",
      all(len(k.split('|')) == 3 and k.split('|')[0] in ('A1', 'A2')
          and k.split('|')[1] in cf.ALL_CONDITIONS and k.split('|')[2] in ('HIGH', 'LOW')
          for k in series_dict))

# ==========================================================================
# 2) DSR MECHANICS SANITY (run_dsr itself is Phase 12e's own already-
# validated module, reused unchanged -- just confirm the split/merge
# plumbing around it works and N reflects the actual trial count).
# ==========================================================================
trial_inputs, summary = run_dsr(series_dict)
check("DSR PLUMBING: trial_inputs has one row per input series",
      len(trial_inputs) == len(series_dict))
check("DSR PLUMBING: N_trials in the summary equals the number of series fed in",
      (summary['N_trials'] == len(series_dict)).all())
check("DSR PLUMBING: SR0 is identical (shared) across every row",
      summary['SR0_daily'].nunique() == 1)

trial_inputs[['asset', 'condition', 'group']] = trial_inputs['asset'].str.split('|', expand=True)
check("DSR PLUMBING: asset|condition|group split recovers valid values for every row",
      trial_inputs['asset'].isin(['A1', 'A2']).all()
      and trial_inputs['condition'].isin(cf.ALL_CONDITIONS).all()
      and trial_inputs['group'].isin(['HIGH', 'LOW']).all())

# ==========================================================================
# 3) POWER CHECK ON expected_max_sharpe/PSR THEMSELVES (Phase 12e's own
# calibration already validated these -- this just re-confirms the
# wiring at a DIFFERENT N, since this step uses N~140 rather than N=14):
# a single genuinely skilled series among many noise series should survive
# DSR; pure noise series should not.
# ==========================================================================
rng2 = np.random.default_rng(99)
noise_series = {f"noise_{i}": rng2.normal(0.0, 0.01, 1000) for i in range(29)}
skilled_series = {"skilled": rng2.normal(0.0012, 0.01, 1000)}  # real, substantial daily edge
mixed = {**noise_series, **skilled_series}
_, mixed_summary = run_dsr(mixed)
check("POWER: a genuinely skilled series (daily mean 0.0012, vol 0.01) embedded "
      "among 29 pure-noise series survives DSR",
      bool(mixed_summary.loc[mixed_summary['asset'] == 'skilled', 'survives_dsr_at_0.95'].iloc[0]))
n_noise_survive = int(mixed_summary.loc[mixed_summary['asset'] != 'skilled', 'survives_dsr_at_0.95'].sum())
check("POWER: none of the 29 pure-noise series survive DSR",
      n_noise_survive == 0)

print("\n" + ("ALL CHECKS PASSED" if ok else "SOME CHECKS FAILED"))

[FAIL] MECHANICS: at most 20 series (2 assets x 5 conditions x 2 groups)
[PASS] MECHANICS: every returned series is NaN-free
[PASS] MECHANICS: series keys are exactly 'asset|condition|group' with a known asset, a known condition, and a valid group
[PASS] DSR PLUMBING: trial_inputs has one row per input series
[PASS] DSR PLUMBING: N_trials in the summary equals the number of series fed in
[PASS] DSR PLUMBING: SR0 is identical (shared) across every row
[PASS] DSR PLUMBING: asset|condition|group split recovers valid values for every row
[PASS] POWER: a genuinely skilled series (daily mean 0.0012, vol 0.01) embedded among 29 pure-noise series survives DSR
[PASS] POWER: none of the 29 pure-noise series survive DSR

SOME CHECKS FAILED


In [ ]:
import shutil
shutil.move('/content/data/raw/condedge_pricepath_mcpt.py', '/content/condedge_pricepath_mcpt.py')

'/content/condedge_pricepath_mcpt.py'

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd

for _d in dict.fromkeys(
    [os.getcwd()] + ([os.path.dirname(os.path.abspath(__file__))] if '__file__' in dir() else [])
):
    if _d not in sys.path:
        sys.path.insert(0, _d)

import condition_features as cf
from trend_meanrev_signal import build_trend_filtered_signal
from condedge_bet_dataset import extract_trades_for_asset
from condedge_pricepath_mcpt import run_full_condition_mcpt, NREPS_REAL_RUN, SEED as MCPT_SEED
from condedge_dsr import build_all_group_daily_returns
from trend_meanrev_dsr import run_dsr

SECTOR_TRI_PATH = 'data/raw/tri_full_panel_10sectors.csv'
MAIN14_TRI_PATH = 'data/raw/tri_full_panel_14assets.csv'
YIELD_PATH = 'data/raw/treasury_yields_fred.csv'

SECTOR_TICKERS = ['XLE', 'XLF', 'XLK', 'XLV', 'XLI', 'XLP', 'XLY', 'XLU', 'XLB', 'XLRE']


def build_combined_panel(sector_panel, main14_panel):
    """Sector tickers (the tested universe) + HYG/LQD (reference columns
    only, for credit_spread -- never treated as tested assets here)."""
    combined = sector_panel.copy()
    combined['HYG'] = main14_panel['HYG'].reindex(sector_panel.index)
    combined['LQD'] = main14_panel['LQD'].reindex(sector_panel.index)
    return combined


if __name__ == '__main__':
    sector_panel = pd.read_csv(SECTOR_TRI_PATH, index_col=0, parse_dates=True)
    main14_panel = pd.read_csv(MAIN14_TRI_PATH, index_col=0, parse_dates=True)
    yields_df = pd.read_csv(YIELD_PATH, index_col=0, parse_dates=True)

    combined_panel = build_combined_panel(sector_panel, main14_panel)
    daily_index = combined_panel.index

    # ---- Step 2 equivalent: build trades + labels for the 10 sector tickers ----
    t0 = time.time()
    all_rows = []
    for asset in SECTOR_TICKERS:
        exec_pos = build_trend_filtered_signal(combined_panel[[asset]])[asset]
        simple_ret = combined_panel[asset].pct_change(fill_method=None)
        all_rows.extend(extract_trades_for_asset(asset, exec_pos, simple_ret, daily_index))
    bets = pd.DataFrame(all_rows)
    bets = bets[bets['decision_date'].notna()].sort_values('t0').reset_index(drop=True)
    print(f"Total completed round-trip trades (10-sector universe): {len(bets)}")

    labeled = cf.label_trades_by_condition(bets, combined_panel, yields_df)
    print(f"Step 2 (labeling) completed in {time.time() - t0:.1f}s")
    for cond in cf.ALL_CONDITIONS:
        counts = labeled[cond].value_counts(dropna=False)
        print(f"  {cond:18s}  HIGH={counts.get(1.0, 0):5d}  LOW={counts.get(-1.0, 0):5d}  "
              f"NaN={counts.get(np.nan, 0) if labeled[cond].isna().any() else 0:5d}")
    os.makedirs('data/raw', exist_ok=True)
    labeled.to_csv('data/raw/condedge_trades_labeled_6cond_10sectors.csv', index=False)

    # ---- Step 4 equivalent: price-path-permutation MCPT, sector tickers only ----
    t0 = time.time()
    mcpt_results = run_full_condition_mcpt(combined_panel, yields_df, assets=SECTOR_TICKERS,
                                            nreps=NREPS_REAL_RUN, seed=MCPT_SEED)
    elapsed = time.time() - t0
    print(f"\nStep 4 (MCPT, NREPS={NREPS_REAL_RUN}) completed in {elapsed:.1f}s "
          f"({len(mcpt_results)} tests)")
    mcpt_results.to_csv('data/raw/condedge_pricepath_mcpt_results_10sectors.csv', index=False)

    sig = mcpt_results[(mcpt_results['p_value_sharpe'] < 0.05) | (mcpt_results['p_value_cum_return'] < 0.05)]
    print(f"{len(sig)} / {len(mcpt_results)} tests nominal p<0.05 (uncorrected):")
    print(sig.sort_values('p_value_sharpe').to_string(index=False))

    # ---- Step 5 equivalent: DSR pooled correction, N=100 (this universe's own pool) ----
    daily_returns_by_trial = build_all_group_daily_returns(combined_panel[SECTOR_TICKERS + ['HYG', 'LQD']], labeled)
    trial_inputs, summary = run_dsr(daily_returns_by_trial)
    trial_inputs[['asset', 'condition', 'group']] = trial_inputs['asset'].str.split('|', expand=True)
    summary[['asset', 'condition', 'group']] = summary['asset'].str.split('|', expand=True)

    combined_verdict = mcpt_results.merge(
        summary[['asset', 'condition', 'group', 'SR0_daily', 'sr_hat_daily', 'sr_hat_annualized', 'psr', 'survives_dsr_at_0.95']],
        on=['asset', 'condition', 'group'], how='left'
    )
    combined_verdict['survives_mcpt'] = (combined_verdict['p_value_sharpe'] < 0.05) | (combined_verdict['p_value_cum_return'] < 0.05)
    combined_verdict['survives_both'] = combined_verdict['survives_mcpt'] & combined_verdict['survives_dsr_at_0.95']

    n_survive_both = int(combined_verdict['survives_both'].sum())
    print(f"\nSR0 (daily, shared across all {len(summary)} trials): {summary['SR0_daily'].iloc[0]:.4f}")
    print(f"{n_survive_both} / {len(combined_verdict)} tests survive BOTH MCPT AND DSR "
          f"(10-sector-universe pool, N={len(summary)})")
    if n_survive_both > 0:
        print(combined_verdict[combined_verdict['survives_both']].to_string(index=False))
    else:
        print("(none)")

    print("\nMCPT-only survivors and their DSR fate:")
    mcpt_only = combined_verdict[combined_verdict['survives_mcpt']].sort_values('psr', ascending=False)
    print(mcpt_only[['asset', 'condition', 'group', 'p_value_sharpe', 'p_value_cum_return',
                      'sr_hat_annualized', 'psr', 'survives_dsr_at_0.95']].to_string(index=False))

    trial_inputs.to_csv('data/raw/condedge_dsr_trial_inputs_10sectors.csv', index=False)
    combined_verdict.to_csv('data/raw/condedge_dsr_combined_verdict_10sectors.csv', index=False)
    print("\nSaved: condedge_trades_labeled_6cond_10sectors.csv, "
          "condedge_pricepath_mcpt_results_10sectors.csv, "
          "condedge_dsr_trial_inputs_10sectors.csv, condedge_dsr_combined_verdict_10sectors.csv "
          "(all in data/raw/)")

Total completed round-trip trades (10-sector universe): 1051
Step 2 (labeling) completed in 5.1s
  vol_regime          HIGH=  505  LOW=  503  NaN=   43
  trend_strength      HIGH=  440  LOW=  568  NaN=   43
  credit_spread       HIGH=  355  LOW=  379  NaN=  317
  rate_environment    HIGH=  409  LOW=  637  NaN=    5
  structural_break    HIGH=  536  LOW=  505  NaN=   10

Step 4 (MCPT, NREPS=30) completed in 240.2s (100 tests)
15 / 100 tests nominal p<0.05 (uncorrected):
asset        condition group  real_ann_sharpe  p_value_sharpe  real_cum_return  p_value_cum_return
  XLF   trend_strength  HIGH         0.588444        0.033333         1.998050            0.033333
  XLF rate_environment  HIGH         0.500067        0.033333         1.452611            0.033333
  XLF structural_break   LOW         0.495939        0.033333         0.918725            0.133333
  XLI       vol_regime   LOW         0.663046        0.033333         1.135287            0.033333
  XLI   trend_strength  HIGH   

In [ ]:
import sys
for mod in ['condedge_pricepath_mcpt', 'condition_features', 'condedge_bet_dataset',
            'trend_meanrev_signal', 'condedge_dsr', 'trend_meanrev_dsr']:
    sys.modules.pop(mod, None)

In [ ]:
# -*- coding: utf-8 -*-
"""
condedge_sector_universe_run.py

Six-Condition Redesign, Step 6: the second, structurally different
universe (10 sector ETFs: XLE/XLF/XLK/XLV/XLI/XLP/XLY/XLU/XLB/XLRE) run
through the IDENTICAL pipeline already built and validated for the
14-asset universe (Steps 2, 4, 5) -- directly testing whether the 14-asset
leg's null is an artifact of that specific universe's own structure, per
the user's own stated reason for wanting a two-universe comparison.

PRE-COMMITTED DESIGN (no new methodology -- this step is deliberately just
re-running the SAME already-validated machinery on new data, not building
anything new):

1. **credit_spread and rate_environment are genuinely external, market-
   wide regime signals** -- they were never conceptually tied to the
   14-asset universe's own constituents, HYG and LQD just happened to
   BOTH be traded assets in that universe AND the credit-spread inputs.
   For the sector-ETF universe, HYG/LQD are NOT among the 10 traded
   assets, so they are pulled in here purely as REFERENCE columns
   (from the already-validated `tri_full_panel_14assets.csv`) alongside
   the 10 sector tickers -- condition_features.py's own functions need no
   change, since they already only ever special-case HYG/LQD by column
   name, not by "is this asset in the tested universe."
2. **Everything else is a literal rerun**: `condition_features.
   label_trades_by_condition` (Step 2), `condedge_pricepath_mcpt.
   run_full_condition_mcpt` (Step 4, same fixed NREPS=30 / seed=20260930
   convention -- see that module's own point 6a for why NREPS must be a
   fixed constant, not a live benchmark), and `condedge_dsr.
   build_all_group_daily_returns` + `run_dsr` (Step 5) -- all imported
   and called unchanged, restricted to the 10 sector tickers.
3. **N for this universe's own DSR pool is 10 x 6 x 2 = 120** (now that
   Step 3's real market_breadth condition is wired in, alongside the
   original 5 -- a separate, smaller pool for a separate universe, not
   merged with the 14-asset universe's own 168 trials, since SR0 is meant
   to correct for the multiple tests actually run WITHIN one universe's
   own search, not across two independently-designed studies).

Input: data/raw/tri_full_panel_10sectors.csv, data/raw/tri_full_panel_14assets.csv
       (for HYG/LQD reference columns only), data/raw/treasury_yields_fred.csv
"""
import os
import sys
import time
import numpy as np
import pandas as pd

for _d in dict.fromkeys(
    [os.getcwd()] + ([os.path.dirname(os.path.abspath(__file__))] if '__file__' in dir() else [])
):
    if _d not in sys.path:
        sys.path.insert(0, _d)

import condition_features as cf
from trend_meanrev_signal import build_trend_filtered_signal
from condedge_bet_dataset import extract_trades_for_asset
from condedge_pricepath_mcpt import run_full_condition_mcpt, NREPS_REAL_RUN, SEED as MCPT_SEED
from condedge_dsr import build_all_group_daily_returns
from trend_meanrev_dsr import run_dsr

SECTOR_TRI_PATH = 'data/raw/tri_full_panel_10sectors.csv'
MAIN14_TRI_PATH = 'data/raw/tri_full_panel_14assets.csv'
YIELD_PATH = 'data/raw/treasury_yields_fred.csv'

SECTOR_TICKERS = ['XLE', 'XLF', 'XLK', 'XLV', 'XLI', 'XLP', 'XLY', 'XLU', 'XLB', 'XLRE']


def build_combined_panel(sector_panel, main14_panel):
    """Sector tickers (the tested universe) + HYG/LQD (reference columns
    only, for credit_spread -- never treated as tested assets here)."""
    combined = sector_panel.copy()
    combined['HYG'] = main14_panel['HYG'].reindex(sector_panel.index)
    combined['LQD'] = main14_panel['LQD'].reindex(sector_panel.index)
    return combined


if __name__ == '__main__':
    sector_panel = pd.read_csv(SECTOR_TRI_PATH, index_col=0, parse_dates=True)
    main14_panel = pd.read_csv(MAIN14_TRI_PATH, index_col=0, parse_dates=True)
    yields_df = pd.read_csv(YIELD_PATH, index_col=0, parse_dates=True)

    # market_breadth (Step 3) is a genuinely EXTERNAL, market-wide S&P 500
    # series -- the SAME file the 14-asset universe run uses, no sector-
    # specific pull needed; it is never specific to which universe is being
    # tested, only to the calendar date.
    if os.path.exists(cf.BREADTH_PATH):
        breadth_series = cf.compute_market_breadth_series(cf.BREADTH_PATH)
        print(f"Loaded real market_breadth series ({cf.BREADTH_PATH}): "
              f"{breadth_series.index.min().date()} to {breadth_series.index.max().date()}")
    else:
        breadth_series = None
        print(f"[market_breadth STILL PLACEHOLDER] {cf.BREADTH_PATH} not found -- "
              f"proceeding with market_breadth as all-NaN for now (5 usable conditions).")

    combined_panel = build_combined_panel(sector_panel, main14_panel)
    daily_index = combined_panel.index

    # ---- Step 2 equivalent: build trades + labels for the 10 sector tickers ----
    t0 = time.time()
    all_rows = []
    for asset in SECTOR_TICKERS:
        exec_pos = build_trend_filtered_signal(combined_panel[[asset]])[asset]
        simple_ret = combined_panel[asset].pct_change(fill_method=None)
        all_rows.extend(extract_trades_for_asset(asset, exec_pos, simple_ret, daily_index))
    bets = pd.DataFrame(all_rows)
    bets = bets[bets['decision_date'].notna()].sort_values('t0').reset_index(drop=True)
    print(f"Total completed round-trip trades (10-sector universe): {len(bets)}")

    labeled = cf.label_trades_by_condition(bets, combined_panel, yields_df, breadth_series=breadth_series)
    print(f"Step 2 (labeling) completed in {time.time() - t0:.1f}s")
    for cond in cf.ALL_CONDITIONS:
        counts = labeled[cond].value_counts(dropna=False)
        print(f"  {cond:18s}  HIGH={counts.get(1.0, 0):5d}  LOW={counts.get(-1.0, 0):5d}  "
              f"NaN={counts.get(np.nan, 0) if labeled[cond].isna().any() else 0:5d}")
    os.makedirs('data/raw', exist_ok=True)
    labeled.to_csv('data/raw/condedge_trades_labeled_6cond_10sectors.csv', index=False)

    # ---- Step 4 equivalent: price-path-permutation MCPT, sector tickers only ----
    t0 = time.time()
    mcpt_results = run_full_condition_mcpt(combined_panel, yields_df, assets=SECTOR_TICKERS,
                                            nreps=NREPS_REAL_RUN, seed=MCPT_SEED,
                                            breadth_series=breadth_series)
    elapsed = time.time() - t0
    print(f"\nStep 4 (MCPT, NREPS={NREPS_REAL_RUN}) completed in {elapsed:.1f}s "
          f"({len(mcpt_results)} tests)")
    mcpt_results.to_csv('data/raw/condedge_pricepath_mcpt_results_10sectors.csv', index=False)

    sig = mcpt_results[(mcpt_results['p_value_sharpe'] < 0.05) | (mcpt_results['p_value_cum_return'] < 0.05)]
    print(f"{len(sig)} / {len(mcpt_results)} tests nominal p<0.05 (uncorrected):")
    print(sig.sort_values('p_value_sharpe').to_string(index=False))

    # ---- Step 5 equivalent: DSR pooled correction, N=100 (this universe's own pool) ----
    daily_returns_by_trial = build_all_group_daily_returns(combined_panel[SECTOR_TICKERS + ['HYG', 'LQD']], labeled)
    trial_inputs, summary = run_dsr(daily_returns_by_trial)
    trial_inputs[['asset', 'condition', 'group']] = trial_inputs['asset'].str.split('|', expand=True)
    summary[['asset', 'condition', 'group']] = summary['asset'].str.split('|', expand=True)

    combined_verdict = mcpt_results.merge(
        summary[['asset', 'condition', 'group', 'SR0_daily', 'sr_hat_daily', 'sr_hat_annualized', 'psr', 'survives_dsr_at_0.95']],
        on=['asset', 'condition', 'group'], how='left'
    )
    combined_verdict['survives_mcpt'] = (combined_verdict['p_value_sharpe'] < 0.05) | (combined_verdict['p_value_cum_return'] < 0.05)
    combined_verdict['survives_both'] = combined_verdict['survives_mcpt'] & combined_verdict['survives_dsr_at_0.95']

    n_survive_both = int(combined_verdict['survives_both'].sum())
    print(f"\nSR0 (daily, shared across all {len(summary)} trials): {summary['SR0_daily'].iloc[0]:.4f}")
    print(f"{n_survive_both} / {len(combined_verdict)} tests survive BOTH MCPT AND DSR "
          f"(10-sector-universe pool, N={len(summary)})")
    if n_survive_both > 0:
        print(combined_verdict[combined_verdict['survives_both']].to_string(index=False))
    else:
        print("(none)")

    print("\nMCPT-only survivors and their DSR fate:")
    mcpt_only = combined_verdict[combined_verdict['survives_mcpt']].sort_values('psr', ascending=False)
    print(mcpt_only[['asset', 'condition', 'group', 'p_value_sharpe', 'p_value_cum_return',
                      'sr_hat_annualized', 'psr', 'survives_dsr_at_0.95']].to_string(index=False))

    trial_inputs.to_csv('data/raw/condedge_dsr_trial_inputs_10sectors.csv', index=False)
    combined_verdict.to_csv('data/raw/condedge_dsr_combined_verdict_10sectors.csv', index=False)
    print("\nSaved: condedge_trades_labeled_6cond_10sectors.csv, "
          "condedge_pricepath_mcpt_results_10sectors.csv, "
          "condedge_dsr_trial_inputs_10sectors.csv, condedge_dsr_combined_verdict_10sectors.csv "
          "(all in data/raw/)")

Loaded real market_breadth series (data/raw/sp500_breadth_pct_above_200dma.csv): 1926-08-30 to 2024-12-31
Total completed round-trip trades (10-sector universe): 1051
Step 2 (labeling) completed in 7.4s
  vol_regime          HIGH=  505  LOW=  503  NaN=   43
  trend_strength      HIGH=  440  LOW=  568  NaN=   43
  credit_spread       HIGH=  355  LOW=  379  NaN=  317
  rate_environment    HIGH=  409  LOW=  637  NaN=    5
  market_breadth      HIGH=  531  LOW=  478  NaN=   42
  structural_break    HIGH=  536  LOW=  505  NaN=   10

Step 4 (MCPT, NREPS=30) completed in 292.6s (120 tests)
20 / 120 tests nominal p<0.05 (uncorrected):
asset        condition group  real_ann_sharpe  p_value_sharpe  real_cum_return  p_value_cum_return
  XLE   market_breadth   LOW         0.301862        0.033333         0.805100            0.033333
  XLF   trend_strength  HIGH         0.588444        0.033333         1.998050            0.033333
  XLF rate_environment  HIGH         0.500067        0.033333       

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import comb
from itertools import combinations
from sklearn.model_selection._split import _BaseKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, accuracy_score, roc_auc_score

import condition_features as cf

N_SPLITS = 6
N_TEST_SPLITS = 2
PURGE = pd.Timedelta(days=252)   # see module docstring point 4 -- NOT step3b's 31 days
EMBARGO = pd.Timedelta(days=252)
RF_PARAMS = dict(n_estimators=500, class_weight='balanced', min_samples_leaf=20,
                  max_features='sqrt', random_state=20260930, n_jobs=-1)
SEED = 20260930

FEATURES = [f'{cond}_rank' for cond in cf.ALL_CONDITIONS]  # 6 features


# ==========================================================================
# Expanding percentile rank -- the continuous generalization of Step 2's
# own expanding_median_label/_sparse (see module docstring point 2).
# ==========================================================================

def expanding_percentile_rank_at_dates(full_daily_series, query_dates, inception,
                                        min_warmup_days=cf.MIN_WARMUP_DAYS):
    """
    full_daily_series : one asset's own full daily condition series (already
        active-window-masked, e.g. cf.build_daily_condition_panels(...)
        [cond][asset]).
    query_dates : iterable of Timestamps (this asset's own trade decision
        dates) to compute a rank at.
    inception : this asset's own TRUE first valid price date (calendar
        warmup gate, same convention as Step 2).

    Returns pd.Series indexed by query_dates: for each date d, the fraction
    of STRICTLY PRIOR valid daily values (positions [0, i-1], i = d's own
    position in full_daily_series) that are <= the value AT d (0.0 = today's
    value is the lowest ever seen so far, 1.0 = the highest, 0.5 = exactly
    at the running median -- the same boundary expanding_median_label's
    +-1.0 call is made at). NaN before the 252-calendar-day-since-inception
    warmup clears, or when today's value or every prior value is missing.
    """
    idx = full_daily_series.index
    vals = full_daily_series.to_numpy()
    out = {}
    for d in query_dates:
        days_since_inception = (d - inception).days
        if days_since_inception < min_warmup_days or d not in idx:
            out[d] = np.nan
            continue
        pos = idx.get_loc(d)
        today_val = vals[pos]
        prior = vals[:pos]
        prior = prior[~np.isnan(prior)]
        if len(prior) == 0 or np.isnan(today_val):
            out[d] = np.nan
            continue
        out[d] = float(np.mean(prior <= today_val))
    return pd.Series(out)


def expanding_percentile_rank_sparse_at_dates(sparse_series, inception,
                                               min_warmup_days=cf.MIN_WARMUP_DAYS):
    """
    sparse_series : one asset's own SADF values, indexed by the SPARSE set
        of dates they were actually evaluated at (e.g.
        cf.compute_structural_break_at_dates's own output).

    Returns a same-indexed Series: the percentile rank of each value within
    STRICTLY PRIOR SPARSE evaluations only -- matching structural_break's
    own already-established sparse-history convention
    (cf.expanding_median_label_sparse), a flagged compute-cost
    simplification carried over unchanged, not a new one.
    """
    sparse_series = sparse_series.sort_index()
    vals = sparse_series.to_numpy()
    n = len(vals)
    ranks = np.full(n, np.nan)
    for i in range(n):
        days_since_inception = (sparse_series.index[i] - inception).days
        if days_since_inception < min_warmup_days:
            continue
        today_val = vals[i]
        prior = vals[:i]
        prior = prior[~np.isnan(prior)]
        if len(prior) == 0 or np.isnan(today_val):
            continue
        ranks[i] = float(np.mean(prior <= today_val))
    return pd.Series(ranks, index=sparse_series.index)


def attach_condition_percentile_ranks(bets, price_panel, yields_df, breadth_series=None):
    """
    bets : Step 2's own labeled-bets DataFrame (or any DataFrame with
        'asset', 'decision_date' columns) -- read-only, not mutated.

    Returns a copy of `bets` with 6 NEW columns, '{condition}_rank' for
    every condition in cf.ALL_CONDITIONS (vol_regime_rank, trend_strength_
    rank, credit_spread_rank, rate_environment_rank, market_breadth_rank,
    structural_break_rank), each a percentile rank in [0, 1] or NaN. Same
    no-lookahead discipline and warmup gate as Step 2's own +-1.0 labels;
    Step 2's own label columns (if present in `bets`) are left untouched.
    """
    bets = bets.copy()
    daily_panels = cf.build_daily_condition_panels(price_panel, yields_df, breadth_series=breadth_series)
    first_valid = price_panel.apply(lambda s: s.first_valid_index())

    for cond in cf.DAILY_CONDITIONS:
        panel = daily_panels[cond]
        rank_col = pd.Series(np.nan, index=bets.index)
        for asset, rows in bets.groupby('asset').groups.items():
            if asset not in panel.columns or first_valid[asset] is None:
                continue
            query_dates = sorted(set(bets.loc[rows, 'decision_date']))
            ranks = expanding_percentile_rank_at_dates(panel[asset], query_dates, first_valid[asset])
            for i in rows:
                d = bets.at[i, 'decision_date']
                if d in ranks.index:
                    rank_col.at[i] = ranks.at[d]
        bets[f'{cond}_rank'] = rank_col.values

    sb_rank_col = pd.Series(np.nan, index=bets.index)
    for asset, rows in bets.groupby('asset').groups.items():
        if asset not in price_panel.columns or first_valid[asset] is None:
            continue
        dates_needed = sorted(set(bets.loc[rows, 'decision_date']))
        sadf_vals = cf.compute_structural_break_at_dates(price_panel[asset], dates_needed)
        sb_ranks = expanding_percentile_rank_sparse_at_dates(sadf_vals, first_valid[asset])
        for i in rows:
            d = bets.at[i, 'decision_date']
            if d in sb_ranks.index:
                sb_rank_col.at[i] = sb_ranks.at[d]
    bets['structural_break_rank'] = sb_rank_col.values

    return bets


# ==========================================================================
# CombPurgedKFoldCV + CV pipeline -- duplicated verbatim from
# step3b_metalabel_final_feature_round.py (this project's own already-
# established meta-labeling architecture -- see module docstring point 3).
# ==========================================================================

class CombPurgedKFoldCV(_BaseKFold):
    def __init__(self, n_splits=5, n_test_splits=2, holding_dates=None,
                 purge=pd.Timedelta(days=0), embargo=pd.Timedelta(days=0),
                 warm_up_end=None, fixed_width=None, safe=False):
        if not isinstance(holding_dates, (pd.Series, pd.DataFrame)):
            raise ValueError('Holding dates must be a pandas series or data frame.')
        elif isinstance(holding_dates, pd.Series):
            holding_dates = pd.DataFrame(holding_dates)
        self.holding_dates = holding_dates.copy()
        if n_test_splits <= 0 or n_test_splits >= n_splits - 1:
            raise ValueError(f'Need 0 < n_test_splits < n_splits - 1. Got n_test_splits = {n_test_splits}.')
        if fixed_width is not None:
            self.holding_dates['t1'] = self.holding_dates['t0'] + fixed_width
        elif 't1' not in self.holding_dates:
            raise ValueError("holding_dates must include a column 't1' or you must specify fixed_width.")
        assert self.holding_dates['t0'].is_monotonic_increasing, \
            "holding_dates must be sorted by t0 -- see step3b's own header note on why"
        self.n_splits = int(n_splits)
        self.n_test_splits = int(n_test_splits)
        self.purge = purge
        self.embargo = embargo
        self.warm_up_end = pd.Timestamp(warm_up_end)
        self.path_count = (n_test_splits / n_splits) * comb(n_splits, n_test_splits)
        self.safe = safe

    def prep_train(self, train_splits, test_splits):
        start_times = [np.min(a) - self.purge for a in test_splits]
        end_times = [np.max(a) + self.embargo + pd.Timedelta(hours=1) for a in test_splits]
        is_bad = pd.Series(False, index=self.holding_dates.index)
        train_dates = [date for a in train_splits for date in a]
        for i in range(len(test_splits)):
            envelopes = (self.holding_dates['t0'] <= start_times[i]) & (self.holding_dates['t1'] >= end_times[i])
            starts_in = self.holding_dates['t0'].between(start_times[i], end_times[i], inclusive='left')
            ends_in = self.holding_dates['t1'].between(start_times[i], end_times[i], inclusive='right')
            is_bad = is_bad | envelopes | starts_in | ends_in
        to_keep = self.holding_dates['t0'].isin(train_dates) & ~is_bad
        return list(self.holding_dates.loc[to_keep, :].index)

    def prep_test(self, test_splits):
        test_idx = np.concatenate([
            self.holding_dates.loc[self.holding_dates['t0'].isin(a), 't0'].index for a in test_splits
        ])
        return test_idx.tolist()

    def split(self, X, y=None, groups=None):
        if np.any(X.index != self.holding_dates.index):
            raise ValueError('X and holding dates must have the same index')
        is_warm_up = self.holding_dates['t0'] <= self.warm_up_end
        splits = np.array_split(self.holding_dates.loc[~is_warm_up, 't0'].unique(), self.n_splits)
        warm_up_dates = list(self.holding_dates.loc[is_warm_up, 't0'].unique())
        splits = [list(a) for a in splits]
        for test_splits in combinations(splits, r=self.n_test_splits):
            train_splits = [warm_up_dates] + [a for a in splits if a not in test_splits]
            train_idx = self.prep_train(train_splits, test_splits)
            test_idx = self.prep_test(test_splits)
            if len(train_idx) == 0 or len(test_idx) == 0:
                continue
            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        if self.safe:
            return len(list(self.split(X)))
        return int(comb(self.n_splits, self.n_test_splits))


def audit_no_leakage(cv, X, holding_dates):
    t0 = holding_dates['t0'].values
    t1 = holding_dates['t1'].values
    n_folds = 0
    for train_idx, test_idx in cv.split(X):
        train_t0, train_t1 = t0[train_idx], t1[train_idx]
        test_t0, test_t1 = t0[test_idx], t1[test_idx]
        overlap = (train_t0[None, :] < test_t1[:, None]) & (train_t1[None, :] > test_t0[:, None])
        assert not overlap.any(), f"Leakage in fold {n_folds}: a training bet's holding window overlaps a test bet's"
        n_folds += 1
    assert n_folds > 0, "CV produced zero usable folds"
    return n_folds


def fold_metrics(y_test, proba, y_train):
    base_rate = y_train.mean()
    base_proba = np.full(len(y_test), base_rate)
    return {
        'n_test': len(y_test),
        'log_loss': log_loss(y_test, proba, labels=[0, 1]),
        'log_loss_baseline': log_loss(y_test, base_proba, labels=[0, 1]),
        'accuracy': accuracy_score(y_test, (proba >= 0.5).astype(int)),
        'accuracy_baseline': max(base_rate, 1 - base_rate),
        'auc': roc_auc_score(y_test, proba) if len(np.unique(y_test)) > 1 else np.nan,
    }


def fold_feature_importance(clf, X_test, y_test, base_log_loss, features, rng):
    out = {}
    for feat in features:
        X_shuf = X_test.copy()
        shuffled_col = X_shuf[feat].to_numpy(copy=True)
        rng.shuffle(shuffled_col)
        X_shuf[feat] = shuffled_col
        proba_shuf = clf.predict_proba(X_shuf)[:, list(clf.classes_).index(1)]
        out[feat] = log_loss(y_test, proba_shuf, labels=[0, 1]) - base_log_loss
    return out


def run_purged_cv(bets, features, n_splits=N_SPLITS, n_test_splits=N_TEST_SPLITS,
                   purge=PURGE, embargo=EMBARGO, rf_params=RF_PARAMS, seed=SEED):
    """
    bets : must have 't0', 't1' (the trade's own real holding window -- used
        DIRECTLY as CombPurgedKFoldCV's holding dates, a simplification over
        step3b, which had to reconstruct an equivalent from its own monthly
        exec_date/next_exec_date; this redesign's bets already carry the
        genuine article), 'label', and every column in `features`.
    """
    valid = bets[features].notna().all(axis=1)
    b = bets.loc[valid].sort_values('t0').reset_index(drop=True)
    n_dropped = (~valid).sum()

    X = b[features]
    y = b['label']
    holding_dates = pd.DataFrame({'t0': b['t0'].values, 't1': b['t1'].values})

    cv = CombPurgedKFoldCV(n_splits=n_splits, n_test_splits=n_test_splits,
                            holding_dates=holding_dates, purge=purge, embargo=embargo)
    n_folds = audit_no_leakage(cv, X, holding_dates)

    rng = np.random.default_rng(seed)
    fold_rows, imp_rows = [], []
    for fold_i, (train_idx, test_idx) in enumerate(cv.split(X)):
        clf = RandomForestClassifier(**rf_params)
        clf.fit(X.iloc[train_idx], y.iloc[train_idx])
        proba = clf.predict_proba(X.iloc[test_idx])[:, list(clf.classes_).index(1)]

        m = fold_metrics(y.iloc[test_idx].values, proba, y.iloc[train_idx].values)
        m['fold'] = fold_i
        fold_rows.append(m)

        mdi = dict(zip(features, clf.feature_importances_))
        mda = fold_feature_importance(clf, X.iloc[test_idx], y.iloc[test_idx].values, m['log_loss'], features, rng)
        for feat in features:
            imp_rows.append({'fold': fold_i, 'feature': feat, 'mdi': mdi[feat], 'mda_logloss_increase': mda[feat]})

    return pd.DataFrame(fold_rows), pd.DataFrame(imp_rows), n_folds, n_dropped


# === RUN_FROM_HERE ===

if __name__ == '__main__':
    import os
    import sys
    for _d in dict.fromkeys(
        [os.getcwd()] + ([os.path.dirname(os.path.abspath(__file__))] if '__file__' in dir() else [])
    ):
        if _d not in sys.path:
            sys.path.insert(0, _d)

    TRI_PATH = 'data/raw/tri_full_panel_14assets.csv'
    YIELD_PATH = 'data/raw/treasury_yields_fred.csv'
    LABELED_PATH = 'data/raw/condedge_trades_labeled_6cond_14assets.csv'

    price_panel = pd.read_csv(TRI_PATH, index_col=0, parse_dates=True)
    yields_df = pd.read_csv(YIELD_PATH, index_col=0, parse_dates=True)
    bets = pd.read_csv(LABELED_PATH, parse_dates=['t0', 't1', 'decision_date'])

    if os.path.exists(cf.BREADTH_PATH):
        breadth_series = cf.compute_market_breadth_series(cf.BREADTH_PATH)
        print(f"Loaded real market_breadth series ({cf.BREADTH_PATH})")
    else:
        breadth_series = None
        print(f"[market_breadth STILL PLACEHOLDER] {cf.BREADTH_PATH} not found -- "
              f"market_breadth_rank will be all-NaN and dropped from this run's feature set below.")

    import time
    t0 = time.time()
    bets_ranked = attach_condition_percentile_ranks(bets, price_panel, yields_df, breadth_series=breadth_series)
    print(f"Percentile-rank features computed for {len(bets_ranked)} trades in {time.time() - t0:.1f}s")

    pd.set_option('display.width', 160)
    print("\n--- Feature missingness (NaN = warmup not cleared / missing input) ---")
    print(bets_ranked[FEATURES].isna().mean().round(3))

    active_features = [f for f in FEATURES if bets_ranked[f].notna().any()]
    dropped_features = [f for f in FEATURES if f not in active_features]
    if dropped_features:
        print(f"\n[FEATURE DROPPED -- entirely NaN, data not yet available] {dropped_features}")
    print(f"Active feature set for this run ({len(active_features)}/6): {active_features}")

    fold_results, importances, n_folds, n_dropped = run_purged_cv(bets_ranked, active_features)
    print(f"\nDropped {n_dropped} bets with any missing active feature (out of {len(bets_ranked)})")
    print(f"Leak audit passed across all {n_folds} CombPurgedKFoldCV paths "
          f"(PURGE=EMBARGO={PURGE.days} days)")

    print("\n--- Per-fold metrics ---")
    print(fold_results.round(4))

    summary = fold_results[['log_loss', 'log_loss_baseline', 'accuracy', 'accuracy_baseline', 'auc']].agg(['mean', 'std'])
    print("\n--- Mean +/- std across folds ---")
    print(summary.round(4))

    n_beats_logloss = (fold_results['log_loss'] < fold_results['log_loss_baseline']).sum()
    n_beats_acc = (fold_results['accuracy'] > fold_results['accuracy_baseline']).sum()
    print(f"\nClassifier beats the naive baseline on log_loss in {n_beats_logloss}/{n_folds} folds, "
          f"on accuracy in {n_beats_acc}/{n_folds} folds")
    print(f"Mean AUC = {fold_results['auc'].mean():.4f} (0.5 = no better than chance)")

    imp_summary = importances.groupby('feature').agg(
        mdi_mean=('mdi', 'mean'), mdi_std=('mdi', 'std'),
        mda_logloss_increase_mean=('mda_logloss_increase', 'mean'),
        mda_logloss_increase_std=('mda_logloss_increase', 'std'),
    ).sort_values('mda_logloss_increase_mean', ascending=False)
    print("\n--- Feature importance (MDI = in-sample, MDA = OOS log-loss increase when shuffled) ---")
    print(imp_summary.round(5))

    os.makedirs('data/raw', exist_ok=True)
    bets_ranked.to_csv('data/raw/condedge_bets_with_ranks_14assets.csv', index=False)
    fold_results.to_csv('data/raw/condedge_metalabel_cv_fold_results_14assets.csv', index=False)
    importances.to_csv('data/raw/condedge_metalabel_cv_feature_importance_14assets.csv', index=False)
    print("\nSaved: data/raw/condedge_bets_with_ranks_14assets.csv, "
          "data/raw/condedge_metalabel_cv_fold_results_14assets.csv, "
          "data/raw/condedge_metalabel_cv_feature_importance_14assets.csv")

Loaded real market_breadth series (data/raw/sp500_breadth_pct_above_200dma.csv)
Percentile-rank features computed for 1309 trades in 6.5s

--- Feature missingness (NaN = warmup not cleared / missing input) ---
vol_regime_rank          0.001
trend_strength_rank      0.001
credit_spread_rank       0.162
rate_environment_rank    0.010
market_breadth_rank      0.025
structural_break_rank    0.011
dtype: float64
Active feature set for this run (6/6): ['vol_regime_rank', 'trend_strength_rank', 'credit_spread_rank', 'rate_environment_rank', 'market_breadth_rank', 'structural_break_rank']

Dropped 258 bets with any missing active feature (out of 1309)
Leak audit passed across all 15 CombPurgedKFoldCV paths (PURGE=EMBARGO=252 days)

--- Per-fold metrics ---
    n_test  log_loss  log_loss_baseline  accuracy  accuracy_baseline     auc  fold
0      360    0.6626             0.6071    0.6194             0.7260  0.5173     0
1      330    0.6663             0.6152    0.5727             0.7237  0.531

In [ ]:
import shutil
shutil.move('/content/data/raw/condedge_sector_universe_run.py', '/content/condedge_sector_universe_run.py')

'/content/condedge_sector_universe_run.py'

In [ ]:
import shutil
shutil.move('/content/data/raw/condedge_metalabel.py', '/content/condedge_metalabel.py')

'/content/condedge_metalabel.py'

In [ ]:
import os
import sys
import time
import pandas as pd

for _d in dict.fromkeys(
    [os.getcwd()] + ([os.path.dirname(os.path.abspath(__file__))] if '__file__' in dir() else [])
):
    if _d not in sys.path:
        sys.path.insert(0, _d)

import condition_features as cf
from condedge_sector_universe_run import build_combined_panel
from condedge_metalabel import (
    attach_condition_percentile_ranks, run_purged_cv, FEATURES, PURGE,
)

SECTOR_TRI_PATH = 'data/raw/tri_full_panel_10sectors.csv'
MAIN14_TRI_PATH = 'data/raw/tri_full_panel_14assets.csv'
YIELD_PATH = 'data/raw/treasury_yields_fred.csv'
LABELED_PATH = 'data/raw/condedge_trades_labeled_6cond_10sectors.csv'

if __name__ == '__main__':
    sector_panel = pd.read_csv(SECTOR_TRI_PATH, index_col=0, parse_dates=True)
    main14_panel = pd.read_csv(MAIN14_TRI_PATH, index_col=0, parse_dates=True)
    yields_df = pd.read_csv(YIELD_PATH, index_col=0, parse_dates=True)
    bets = pd.read_csv(LABELED_PATH, parse_dates=['t0', 't1', 'decision_date'])

    combined_panel = build_combined_panel(sector_panel, main14_panel)

    if os.path.exists(cf.BREADTH_PATH):
        breadth_series = cf.compute_market_breadth_series(cf.BREADTH_PATH)
        print(f"Loaded real market_breadth series ({cf.BREADTH_PATH})")
    else:
        breadth_series = None
        print(f"[market_breadth STILL PLACEHOLDER] {cf.BREADTH_PATH} not found -- "
              f"market_breadth_rank will be all-NaN and dropped from this run's feature set below.")

    t0 = time.time()
    bets_ranked = attach_condition_percentile_ranks(bets, combined_panel, yields_df, breadth_series=breadth_series)
    print(f"Percentile-rank features computed for {len(bets_ranked)} trades in {time.time() - t0:.1f}s")

    pd.set_option('display.width', 160)
    print("\n--- Feature missingness (NaN = warmup not cleared / missing input) ---")
    print(bets_ranked[FEATURES].isna().mean().round(3))

    active_features = [f for f in FEATURES if bets_ranked[f].notna().any()]
    dropped_features = [f for f in FEATURES if f not in active_features]
    if dropped_features:
        print(f"\n[FEATURE DROPPED -- entirely NaN, data not yet available] {dropped_features}")
    print(f"Active feature set for this run ({len(active_features)}/6): {active_features}")

    fold_results, importances, n_folds, n_dropped = run_purged_cv(bets_ranked, active_features)
    print(f"\nDropped {n_dropped} bets with any missing active feature (out of {len(bets_ranked)})")
    print(f"Leak audit passed across all {n_folds} CombPurgedKFoldCV paths "
          f"(PURGE=EMBARGO={PURGE.days} days)")

    print("\n--- Per-fold metrics ---")
    print(fold_results.round(4))

    summary = fold_results[['log_loss', 'log_loss_baseline', 'accuracy', 'accuracy_baseline', 'auc']].agg(['mean', 'std'])
    print("\n--- Mean +/- std across folds ---")
    print(summary.round(4))

    n_beats_logloss = (fold_results['log_loss'] < fold_results['log_loss_baseline']).sum()
    n_beats_acc = (fold_results['accuracy'] > fold_results['accuracy_baseline']).sum()
    print(f"\nClassifier beats the naive baseline on log_loss in {n_beats_logloss}/{n_folds} folds, "
          f"on accuracy in {n_beats_acc}/{n_folds} folds")
    print(f"Mean AUC = {fold_results['auc'].mean():.4f} (0.5 = no better than chance)")

    imp_summary = importances.groupby('feature').agg(
        mdi_mean=('mdi', 'mean'), mdi_std=('mdi', 'std'),
        mda_logloss_increase_mean=('mda_logloss_increase', 'mean'),
        mda_logloss_increase_std=('mda_logloss_increase', 'std'),
    ).sort_values('mda_logloss_increase_mean', ascending=False)
    print("\n--- Feature importance (MDI = in-sample, MDA = OOS log-loss increase when shuffled) ---")
    print(imp_summary.round(5))

    os.makedirs('data/raw', exist_ok=True)
    bets_ranked.to_csv('data/raw/condedge_bets_with_ranks_10sectors.csv', index=False)
    fold_results.to_csv('data/raw/condedge_metalabel_cv_fold_results_10sectors.csv', index=False)
    importances.to_csv('data/raw/condedge_metalabel_cv_feature_importance_10sectors.csv', index=False)
    print("\nSaved: data/raw/condedge_bets_with_ranks_10sectors.csv, "
          "data/raw/condedge_metalabel_cv_fold_results_10sectors.csv, "
          "data/raw/condedge_metalabel_cv_feature_importance_10sectors.csv")

Loaded real market_breadth series (data/raw/sp500_breadth_pct_above_200dma.csv)
Percentile-rank features computed for 1051 trades in 9.0s

--- Feature missingness (NaN = warmup not cleared / missing input) ---
vol_regime_rank          0.000
trend_strength_rank      0.000
credit_spread_rank       0.263
rate_environment_rank    0.005
market_breadth_rank      0.040
structural_break_rank    0.010
dtype: float64
Active feature set for this run (6/6): ['vol_regime_rank', 'trend_strength_rank', 'credit_spread_rank', 'rate_environment_rank', 'market_breadth_rank', 'structural_break_rank']

Dropped 324 bets with any missing active feature (out of 1051)
Leak audit passed across all 15 CombPurgedKFoldCV paths (PURGE=EMBARGO=252 days)

--- Per-fold metrics ---
    n_test  log_loss  log_loss_baseline  accuracy  accuracy_baseline     auc  fold
0      269    0.6303             0.5462    0.6989             0.6986  0.4959     0
1      232    0.6534             0.6105    0.6164             0.7300  0.507

In [ ]:
# -*- coding: utf-8 -*-
"""
condedge_dsr.py

Six-Condition Redesign, Step 5: the Deflated Sharpe Ratio (DSR) pooled
correction across all 140 (asset, condition, group) tests from Step 4 --
this project's own standing rule ("an edge is only accepted as real if
BOTH the permutation test AND the DSR are significant") applied to the
14-asset universe leg of this redesign, exactly as it was applied to
trend-filtered mean-reversion itself in Phase 12d/12e.

PRE-COMMITTED DESIGN:

1. REUSES trend_meanrev_dsr.py's `run_dsr` UNCHANGED (Bailey & Lopez de
   Prado 2014: extreme-value-theory expected-max-Sharpe sets SR0 across
   all N trials; PSR per trial against SR0, using each trial's own T,
   skewness, and non-excess kurtosis; daily frequency throughout, never
   annualized inside the formula). N = 140 here (14 assets x 5 conditions
   x 2 groups), the SAME trials Step 4 already tested via MCPT.

2. THE 140 "DAILY RETURN SERIES" ARE THE IDENTICAL group-restricted P&L
   series Step 4 already computed its Sharpe/cum-return statistics from
   (condedge_pricepath_mcpt.build_group_position, masked to the standing
   252-day tradeable window) -- built here directly from REAL (unpermuted)
   data only; no permutation involved in this step, DSR is a single-draw
   correction, not a Monte Carlo one. Flat (non-trading) days inside the
   tradeable window are kept as legitimate zero-return observations, not
   dropped -- consistent with how Phase 12d/12e's own per-asset series
   (also mostly-flat) were treated, and with Step 4's own MCPT statistic
   computation on this exact series.

3. FLAGGED LIMITATION, stated plainly rather than glossed over: the
   standard N-trial expected-max-Sharpe correction implicitly treats the
   N trials as (at least approximately) separate draws. Many of these 140
   are NOT independent by construction -- HIGH and LOW within one
   condition are COMPLEMENTARY partitions of the same asset's own trades,
   and the same asset's own trades recur across all 5 conditions. This
   does not invalidate the calculation (SR0 still uses whatever ACTUAL
   cross-sectional dispersion these 140 trial Sharpes exhibit, which
   already reflects their real correlation structure to some extent), but
   it means "N=140" overstates the count of genuinely distinct things
   tried, in the same direction Phase 12e's own SR0 discussion already
   flagged for a cleaner N=14 case (a wider spread of trial Sharpes pushes
   SR0 up). Reported here as an honest caveat on the correction's own
   precision, not a reason to skip the correction.

4. FINAL VERDICT PER TEST: survives MCPT (nominal p<0.05 on Sharpe or cum.
   return, from Step 4's saved results) AND survives DSR (PSR >= 0.95
   against the shared N=140 SR0) -- this project's standing two-part rule,
   unchanged from Phase 12d/12e and momentum's own Phase 6a/6b precedent.

Input: data/raw/condedge_trades_labeled_6cond_14assets.csv,
       data/raw/condedge_pricepath_mcpt_results_14assets.csv,
       data/raw/tri_full_panel_14assets.csv
"""
import numpy as np
import pandas as pd

import condition_features as cf
from trend_meanrev_signal import build_trend_filtered_signal
from trend_meanrev_dsr import run_dsr
from condedge_pricepath_mcpt import build_group_position, GROUPS


def build_all_group_daily_returns(price_panel, labeled_bets):
    """
    labeled_bets : the Step 2 output (one row per completed trade, an
        'asset' column, and the 5 condition HIGH/LOW columns) for the REAL
        14-asset universe.

    Returns dict {"{asset}|{condition}|{group}": np.ndarray} -- the SAME
    group-restricted, tradeable-window-masked daily P&L series Step 4's
    MCPT used for its own real (trial-0) statistic, for all (asset,
    condition, group) combinations with at least one trade in that group.
    """
    daily_index = price_panel.index
    out = {}
    for asset, bets_asset in labeled_bets.groupby('asset'):
        if asset not in price_panel.columns:
            continue
        executed_position = build_trend_filtered_signal(price_panel[[asset]])[asset]
        simple_ret = price_panel[asset].pct_change(fill_method=None)
        trend_valid = price_panel[asset].pct_change(cf.TREND_LOOKBACK, fill_method=None).notna()

        for cond in cf.ALL_CONDITIONS:
            for group_name, label_value in GROUPS.items():
                gp = build_group_position(bets_asset, executed_position, daily_index, cond, label_value)
                gpnl = (gp * simple_ret).where(trend_valid).dropna()
                key = f"{asset}|{cond}|{group_name}"
                out[key] = gpnl.to_numpy()
    return out


# === RUN_FROM_HERE ===

if __name__ == '__main__':
    import os
    import sys
    for _d in dict.fromkeys(
        [os.getcwd()] + ([os.path.dirname(os.path.abspath(__file__))] if '__file__' in dir() else [])
    ):
        if _d not in sys.path:
            sys.path.insert(0, _d)

    TRI_PATH = 'data/raw/tri_full_panel_14assets.csv'
    LABELED_PATH = 'data/raw/condedge_trades_labeled_6cond_14assets.csv'
    MCPT_PATH = 'data/raw/condedge_pricepath_mcpt_results_14assets.csv'

    price_panel = pd.read_csv(TRI_PATH, index_col=0, parse_dates=True)
    labeled_bets = pd.read_csv(LABELED_PATH, parse_dates=['t0', 't1', 'decision_date'])
    mcpt_results = pd.read_csv(MCPT_PATH)

    daily_returns_by_trial = build_all_group_daily_returns(price_panel, labeled_bets)
    print(f"Built {len(daily_returns_by_trial)} group-restricted daily return series "
          f"(expect up to 140 = 14 assets x 5 conditions x 2 groups)")

    trial_inputs, summary = run_dsr(daily_returns_by_trial)
    trial_inputs[['asset', 'condition', 'group']] = trial_inputs['asset'].str.split('|', expand=True)
    summary[['asset', 'condition', 'group']] = summary['asset'].str.split('|', expand=True)

    pd.set_option('display.width', 160)
    n_survive_dsr = int(summary['survives_dsr_at_0.95'].sum())
    print(f"\nSR0 (daily, shared across all {len(summary)} trials): {summary['SR0_daily'].iloc[0]:.4f}")
    print(f"{n_survive_dsr} / {len(summary)} tests survive DSR alone (PSR >= 0.95 against SR0)")

    # Combine with Step 4's own MCPT verdict -- this project's standing
    # two-part rule: BOTH must be significant.
    combined = mcpt_results.merge(
        summary[['asset', 'condition', 'group', 'SR0_daily', 'sr_hat_daily', 'sr_hat_annualized', 'psr', 'survives_dsr_at_0.95']],
        on=['asset', 'condition', 'group'], how='left'
    )
    combined['survives_mcpt'] = (combined['p_value_sharpe'] < 0.05) | (combined['p_value_cum_return'] < 0.05)
    combined['survives_both'] = combined['survives_mcpt'] & combined['survives_dsr_at_0.95']

    n_survive_both = int(combined['survives_both'].sum())
    print(f"\n{n_survive_both} / {len(combined)} tests survive BOTH MCPT (nominal p<0.05) "
          f"AND DSR (PSR>=0.95) -- this project's standing two-part rule for accepting an edge as real.")
    if n_survive_both > 0:
        print(combined[combined['survives_both']].to_string(index=False))
    else:
        print("(none)")

    print(f"\nFor reference, MCPT-only survivors and their DSR fate:")
    mcpt_only = combined[combined['survives_mcpt']].sort_values('psr', ascending=False)
    print(mcpt_only[['asset', 'condition', 'group', 'p_value_sharpe', 'p_value_cum_return',
                      'sr_hat_annualized', 'psr', 'survives_dsr_at_0.95']].to_string(index=False))

    os.makedirs('data/raw', exist_ok=True)
    trial_inputs.to_csv('data/raw/condedge_dsr_trial_inputs.csv', index=False)
    combined.to_csv('data/raw/condedge_dsr_combined_verdict_14assets.csv', index=False)
    print("\nSaved: data/raw/condedge_dsr_trial_inputs.csv, data/raw/condedge_dsr_combined_verdict_14assets.csv")

Built 168 group-restricted daily return series (expect up to 140 = 14 assets x 5 conditions x 2 groups)

SR0 (daily, shared across all 168 trials): 0.0434
0 / 168 tests survive DSR alone (PSR >= 0.95 against SR0)

0 / 168 tests survive BOTH MCPT (nominal p<0.05) AND DSR (PSR>=0.95) -- this project's standing two-part rule for accepting an edge as real.
(none)

For reference, MCPT-only survivors and their DSR fate:
asset        condition group  p_value_sharpe  p_value_cum_return  sr_hat_annualized      psr  survives_dsr_at_0.95
  SPY       vol_regime   LOW        0.033333            0.033333           0.746916 0.629632                 False
  VNQ       vol_regime   LOW        0.033333            0.066667           0.670288 0.464510                 False
  SPY   trend_strength  HIGH        0.033333            0.033333           0.640039 0.385513                 False
  EEM structural_break  HIGH        0.033333            0.033333           0.623021 0.373111                 False
  VNQ  